In [ ]:
# INTRO: 
# This notebook delivers the data engineering and analytical preparation phase of the project.
# The output of this phase will be a set of structured and reusable analytical tables, exported into a dedicated `cleaned_data/` folder and ready for downstream aggregation and comparative analysis.

In [ ]:
#***************************************************************************************************************************
# DEV ENVIRONMENT SETUP
#***************************************************************************************************************************

In [ ]:
#=====================================================
# Install libraries
#=====================================================

In [2]:
!pip install pandas requests duckdb openpyxl matplotlib seaborn numpy dotenv

In [3]:
#=====================================================
# Import libraries
#=====================================================

In [25]:
import pandas as pd
import requests
import duckdb # in-process analytical SQL database designed for fast analytical queries on large datasets - EX: fast SQL queries directly on Pandas DataFrames
import zipfile
import os # interact with operating system - EX: access environement variables

In [26]:
#=====================================================
# Get private data
#=====================================================
from dotenv import load_dotenv

load_dotenv(override=True)

ATMO_FRANCE_BASE_URL = os.getenv("ATMO_FRANCE_BASE_URL", "https://api.atmo-france.org")
ATMO_FRANCE_USERNAME = os.getenv("ATMO_FRANCE_USERNAME")
ATMO_FRANCE_PASSWORD = os.getenv("ATMO_FRANCE_PASSWORD")

if not ATMO_FRANCE_USERNAME or not ATMO_FRANCE_PASSWORD:
    raise ValueError("Variables d'environnement ATMO France manquantes dans le fichier .env.")

In [27]:
#=====================================================
# Generic functions for data collection
#=====================================================

In [28]:
# function to download dataset into browser
def download(url, filename):
    try:
        response = requests.get(url)
        response.raise_for_status()
        
        with open(filename, "wb") as f:
            f.write(response.content)
            
        print("Download successful:", filename)
        
    except Exception as e:
        print("Error:", e)

# function to download zipped dataset into browser and extract files in a specified folder
def download_and_extract_zip(url, zip_path, extract_path):
    try:
        # download
        response = requests.get(url)
        with open(zip_path, "wb") as f:
            f.write(response.content)
        
        # create extract path if not existing
        os.makedirs(extract_path, exist_ok=True) # do not raise exception if folder already exists
        
        # extract files
        with zipfile.ZipFile(zip_path, 'r') as zip_ref: # open zip file in read mode
            zip_ref.extractall(extract_path) #if files already exist in extract path, they are replaced
            
    except Exception as e:
        print("Error:", e)

    print("Download and extraction complete")

# Possible improvement for big files
# with requests.get(url, stream=True) as response: #stream=True downloads the data in chunks - Essential for large files - Prevents memory overload
#    response.raise_for_status()
#    with open(filename, "wb") as f:
#        for chunk in response.iter_content(chunk_size=8192): #chunk_size=8192 → 8 KB per chunk
#            f.write(chunk)

In [29]:
#=====================================================
# Generic function to move columns in a DataFrame
#=====================================================

In [30]:
def move_columns(df, cols_to_move, after=None, before=None):
    """
    Move one or more columns within a DataFrame

    Parameters:
    - df : pandas DataFrame
    - cols_to_move : list columns to move
    - after : the column after which to insert
    - before : the column before which to insert

    Returns:
    - DataFrame with reordered columns
    """

    cols = list(df.columns) # stores initial DataFrame column names in a list

    # secure it : Convert to a list if a single column_to_move is provided
    if isinstance(cols_to_move, str): #Is this variable a string?
        cols_to_move = [cols_to_move] #transform in a list

    # removes the columns to move
    for col in cols_to_move:
        if col in cols:
            cols.remove(col)

    # determines position
    if after:
        pos = cols.index(after) + 1
    elif before:
        pos = cols.index(before)
    else:
        pos = len(cols)  # à la fin

    # insert columns
    cols[pos:pos] = cols_to_move

    return df[cols]

In [31]:
#***************************************************************************************************************************
# QUALITY OF LIFE IN OUR CITIES / QUALITE DE VIE DANS NOS CITEES
#***************************************************************************************************************************

In [32]:
#=====================================================
# I - STANDARD OF LIVING / NIVEAU DE VIE
#=====================================================

In [33]:
#-----------------------------------------------------
# I.a - INCOME analysis / Analyse des SALAIRES
#-----------------------------------------------------

In [35]:
# A - Download zip file and extract datasets
# --------------------------------------------
# A.1 - Declared Incomes
url1 = "https://www.insee.fr/fr/statistiques/fichier/8229323/BASE_TD_FILO_IRIS_2021_DEC_CSV.zip"
download_and_extract_zip(url1, "source_files/Standard_of_living/Income/BASE_TD_FILO_IRIS_2021_DEC_CSV.zip", "source_files/Standard_of_living/Income/Declared_income_2021")
# A.2 - Disposable Incomes
url2 = "https://www.insee.fr/fr/statistiques/fichier/8229323/BASE_TD_FILO_IRIS_2021_DISP_CSV.zip"
download_and_extract_zip(url2, "source_files/Standard_of_living/Income/BASE_TD_FILO_IRIS_2021_DISP_CSV.zip", "source_files/Standard_of_living/Income/Disposable_income_2021")

Download and extraction complete
Download and extraction complete


In [36]:
# A.3 - INSEE codes for locations
url3 = "https://www.insee.fr/fr/statistiques/fichier/8377162/cog_ensemble_2025_csv.zip"
download_and_extract_zip(url3, "source_files/INSEE_general/cog_ensemble_2025_csv.zip", "source_files/INSEE_general")

Download and extraction complete


In [37]:
# B.1 - Read dataset Declared Incomes
declared_income_2021 = pd.read_csv("source_files/Standard_of_living/Income/Declared_income_2021/BASE_TD_FILO_IRIS_2021_DEC.csv", sep=";") # be careful, separator is ;
declared_income_2021.head()

,IRIS,DEC_PIMP21,DEC_TP6021,DEC_INCERT21,DEC_Q121,DEC_MED21,DEC_Q321,DEC_EQ21,DEC_D121,DEC_D221,...,DEC_RD21,DEC_S80S2021,DEC_GI21,DEC_PACT21,DEC_PTSA21,DEC_PCHO21,DEC_PBEN21,DEC_PPEN21,DEC_PAUT21,DEC_NOTE21
0,010040101,"43,0","29,0",2,12610,19330,26390,"0,71",7760,11300,...,"4,4","6,3","0,318","69,3","62,5","3,2","3,6","27,1","3,6",0
1,010040102,"42,0","39,0",1,9730,16830,24620,"0,89",4680,8600,...,"7,1","9,0","0,362","70,5","63,7","4,4","2,4","25,6","3,9",0
2,010040201,"47,0","29,0",1,12220,19940,27650,"0,77",6500,10300,...,"5,7","8,0","0,352","69,3","61,9","3,6","3,8","26,7","4,0",0
3,010040202,"62,0","14,0",1,18350,25560,35010,"0,65",11960,16450,...,"3,8","6,3","0,360","65,5","60,0","2,3","3,2","21,8","12,7",0
4,010330102,"42,0","31,0",1,11280,19870,30050,"0,94",5250,9790,...,"8,5","10,6","0,390","74,1","67,5","4,8","1,8","23,0","2,9",0


In [38]:
declared_income_2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 16026 entries, 0 to 16025
Data columns (total 26 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   IRIS          16026 non-null  str  
 1   DEC_PIMP21    16026 non-null  str  
 2   DEC_TP6021    16026 non-null  str  
 3   DEC_INCERT21  16026 non-null  str  
 4   DEC_Q121      16026 non-null  str  
 5   DEC_MED21     16026 non-null  str  
 6   DEC_Q321      16026 non-null  str  
 7   DEC_EQ21      16026 non-null  str  
 8   DEC_D121      16026 non-null  str  
 9   DEC_D221      16026 non-null  str  
 10  DEC_D321      16026 non-null  str  
 11  DEC_D421      16026 non-null  str  
 12  DEC_D621      16026 non-null  str  
 13  DEC_D721      16026 non-null  str  
 14  DEC_D821      16026 non-null  str  
 15  DEC_D921      16026 non-null  str  
 16  DEC_RD21      16026 non-null  str  
 17  DEC_S80S2021  16026 non-null  str  
 18  DEC_GI21      16026 non-null  str  
 19  DEC_PACT21    16026 non-null  str  


In [39]:
# INITIAL OBSERVATION : all fields are stored as strings, including numeric ones. No missing String values but need conversion to double check.

In [40]:
# B.2 - Read dataset Disposable Incomes
disposable_income_2021 = pd.read_csv("source_files/Standard_of_living/Income/Disposable_income_2021/BASE_TD_FILO_IRIS_2021_DISP.csv", sep=";")
disposable_income_2021.head()

,IRIS,DISP_TP6021,DISP_INCERT21,DISP_Q121,DISP_MED21,DISP_Q321,DISP_EQ21,DISP_D121,DISP_D221,DISP_D321,...,DISP_PCHO21,DISP_PBEN21,DISP_PPEN21,DISP_PPAT21,DISP_PPSOC21,DISP_PPFAM21,DISP_PPMINI21,DISP_PPLOGT21,DISP_PIMPOT21,DISP_NOTE21
0,010040101,"19,0",2,14990,20350,26140,"0,55",11620,14280,16080,...,"3,0","3,6","26,9","6,2","8,6","3,3","3,8","1,5","-12,5",0
1,010040102,"25,0",1,13880,18570,24760,"0,59",10580,12890,14660,...,"4,2","2,4","24,9","5,8","11,1","3,7","5,1","2,3","-12,4",0
2,010040201,"19,0",1,15190,20700,27180,"0,58",11400,14060,16320,...,"3,5","4,0","27,2","6,4","7,7","2,8","3,3","1,6","-13,8",0
3,010040202,"8,0",1,19600,25230,33170,"0,54",14810,18310,20780,...,"2,4","3,6","23,8","16,2","4,0","1,8","1,5","0,7","-17,3",0
4,010330102,"24,0",1,14050,20420,29640,"0,76",9410,12570,15130,...,"4,9","1,9","23,7","5,2","5,3","1,5","2,5","1,3","-13,1",0


In [41]:
disposable_income_2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 16026 entries, 0 to 16025
Data columns (total 30 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   IRIS           16026 non-null  str  
 1   DISP_TP6021    16026 non-null  str  
 2   DISP_INCERT21  16026 non-null  str  
 3   DISP_Q121      16026 non-null  str  
 4   DISP_MED21     16026 non-null  str  
 5   DISP_Q321      16026 non-null  str  
 6   DISP_EQ21      16026 non-null  str  
 7   DISP_D121      16026 non-null  str  
 8   DISP_D221      16026 non-null  str  
 9   DISP_D321      16026 non-null  str  
 10  DISP_D421      16026 non-null  str  
 11  DISP_D621      16026 non-null  str  
 12  DISP_D721      16026 non-null  str  
 13  DISP_D821      16026 non-null  str  
 14  DISP_D921      16026 non-null  str  
 15  DISP_RD21      16026 non-null  str  
 16  DISP_S80S2021  16026 non-null  str  
 17  DISP_GI21      16026 non-null  str  
 18  DISP_PACT21    16026 non-null  str  
 19  DISP_PTSA21    

In [42]:
# INITIAL OBSERVATION : all fields are stored as strings, including numeric ones. No missing String values but need conversion to double check.

In [43]:
# B.3 - Read dataset INSEE code to commune name

In [44]:
insee_to_commune = pd.read_csv("source_files/INSEE_general/v_commune_2025.csv", sep=",")
insee_to_commune.head() 

,TYPECOM,COM,REG,DEP,CTCD,ARR,TNCC,NCC,NCCENR,LIBELLE,CAN,COMPARENT
0,COM,01001,84.0,01,01D,012,5,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,L'Abergement-Clémenciat,0108,NaN
1,COM,01002,84.0,01,01D,011,5,ABERGEMENT DE VAREY,Abergement-de-Varey,L'Abergement-de-Varey,0101,NaN
2,COM,01004,84.0,01,01D,011,1,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,Ambérieu-en-Bugey,0101,NaN
3,COM,01005,84.0,01,01D,012,1,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,Ambérieux-en-Dombes,0122,NaN
4,COM,01006,84.0,01,01D,011,1,AMBLEON,Ambléon,Ambléon,0104,NaN


In [45]:
# Only keep the real commune names: filter on "TYPECOM" = "COM"
insee_to_commune = insee_to_commune[
    insee_to_commune["TYPECOM"] == "COM"
].copy()

In [48]:
# B.4 - Read dataset IRIS code to precise location in commune
iris_to_commune = pd.read_csv("source_files/Standard_of_living/Income/Disposable_income_2021/meta_BASE_TD_FILO_IRIS_2021_DISP.csv", sep=";")
iris_to_commune.head() 

,COD_VAR,LIB_VAR,LIB_VAR_LONG,COD_MOD,LIB_MOD,TYPE_VAR,LONG_VAR
0,IRIS,IRIS,Code du département suivi du numéro de commune...,NaN,NaN,CHAR,9
1,DISP_TP6021,Taux de pauvreté au seuil de 60 % (%),Taux de pauvreté au seuil de 60 % du revenu di...,NaN,NaN,CHAR,4
2,DISP_INCERT21,Incertitude sur les indicateurs DISP_TP6021,Incertitude sur le taux de bas revenus,1,IRIS de 1 000 à moins de 10 000 ménages : à l’...,CHAR,2
3,DISP_INCERT21,Incertitude sur les indicateurs DISP_TP6021,Incertitude sur le taux de bas revenus,2,IRIS de 500 à moins de 1 000 ménages : à l’ent...,CHAR,2
4,DISP_Q121,1ᵉʳ quartile (€),1ᵉʳ quartile du revenu disponible par unité de...,NaN,NaN,CHAR,5


In [49]:
# C - Clean dataset 

In [50]:
#C.1.1 declared_income_2021 - Keep only needed columns

In [51]:
declared_income_2021.drop(
    columns=[
        "DEC_INCERT21","DEC_D221","DEC_D321","DEC_D421","DEC_D621","DEC_D721","DEC_D821",
        "DEC_EQ21","DEC_RD21","DEC_S80S2021","DEC_GI21",
        "DEC_PTSA21","DEC_PCHO21","DEC_PBEN21","DEC_NOTE21"
    ],
    inplace=True
)
declared_income_2021.head()

,IRIS,DEC_PIMP21,DEC_TP6021,DEC_Q121,DEC_MED21,DEC_Q321,DEC_D121,DEC_D921,DEC_PACT21,DEC_PPEN21,DEC_PAUT21
0,010040101,"43,0","29,0",12610,19330,26390,7760,33850,"69,3","27,1","3,6"
1,010040102,"42,0","39,0",9730,16830,24620,4680,33030,"70,5","25,6","3,9"
2,010040201,"47,0","29,0",12220,19940,27650,6500,37220,"69,3","26,7","4,0"
3,010040202,"62,0","14,0",18350,25560,35010,11960,45520,"65,5","21,8","12,7"
4,010330102,"42,0","31,0",11280,19870,30050,5250,44820,"74,1","23,0","2,9"


In [52]:
#C.2.1 disposable_income_2021 - Keep only needed columns

In [53]:
disposable_income_2021.drop(
    columns=[
        "DISP_INCERT21","DISP_D221","DISP_D321","DISP_D421","DISP_D621","DISP_D721","DISP_D821",
        "DISP_EQ21","DISP_RD21","DISP_S80S2021","DISP_GI21",
        "DISP_PTSA21","DISP_PCHO21","DISP_PBEN21","DISP_NOTE21"
    ],
    inplace=True
)
disposable_income_2021.head()

,IRIS,DISP_TP6021,DISP_Q121,DISP_MED21,DISP_Q321,DISP_D121,DISP_D921,DISP_PACT21,DISP_PPEN21,DISP_PPAT21,DISP_PPSOC21,DISP_PPFAM21,DISP_PPMINI21,DISP_PPLOGT21,DISP_PIMPOT21
0,010040101,"19,0",14990,20350,26140,11620,32060,"70,8","26,9","6,2","8,6","3,3","3,8","1,5","-12,5"
1,010040102,"25,0",13880,18570,24760,10580,31130,"70,6","24,9","5,8","11,1","3,7","5,1","2,3","-12,4"
2,010040201,"19,0",15190,20700,27180,11400,34450,"72,5","27,2","6,4","7,7","2,8","3,3","1,6","-13,8"
3,010040202,"8,0",19600,25230,33170,14810,41230,"73,3","23,8","16,2","4,0","1,8","1,5","0,7","-17,3"
4,010330102,"24,0",14050,20420,29640,9410,42390,"78,9","23,7","5,2","5,3","1,5","2,5","1,3","-13,1"


In [54]:
#C.3.1 insee_to_commune - Keep only needed columns

In [55]:
insee_to_commune.drop(
    columns=[
        "TYPECOM","REG","DEP","CTCD","ARR","TNCC","LIBELLE","CAN","COMPARENT"
    ],
    inplace=True
)
insee_to_commune.head()

,COM,NCC,NCCENR
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes
4,01006,AMBLEON,Ambléon


In [56]:
#C.4.1 iris_to_commune - Keep only needed columns and lines

In [57]:
iris_to_commune.drop(
    columns=[
        "COD_VAR","LIB_VAR","LIB_VAR_LONG","TYPE_VAR","LONG_VAR"
    ],
    inplace=True
)
iris_to_commune.head()

,COD_MOD,LIB_MOD
0,NaN,NaN
1,NaN,NaN
2,1,IRIS de 1 000 à moins de 10 000 ménages : à l’...
3,2,IRIS de 500 à moins de 1 000 ménages : à l’ent...
4,NaN,NaN


In [58]:
# iris_to_commune - Remove empty lines

In [59]:
iris_to_commune = iris_to_commune.dropna()
iris_to_commune.head(10)

,COD_MOD,LIB_MOD
2,1,IRIS de 1 000 à moins de 10 000 ménages : à l’...
3,2,IRIS de 500 à moins de 1 000 ménages : à l’ent...
30,0,Aucun problème particulier. Certaines données ...
31,1,IRIS pour lesquels l’évolution du nombre de mé...
32,5,Données non diffusées en raison d'anomalies re...
33,010040101,Les Pérouses-Triangle d'Activités
34,010040102,Longeray-Gare
35,010040201,Centre-Saint-Germain-Vareilles
36,010040202,Tiret-Les Allymes
37,010330102,Centre Ville


In [60]:
# iris_to_commune - Remove comment lines and reset index

In [61]:
iris_to_commune = iris_to_commune[iris_to_commune.index >= 33]
iris_to_commune.reset_index(drop=True, inplace=True)
iris_to_commune.head()

,COD_MOD,LIB_MOD
0,010040101,Les Pérouses-Triangle d'Activités
1,010040102,Longeray-Gare
2,010040201,Centre-Saint-Germain-Vareilles
3,010040202,Tiret-Les Allymes
4,010330102,Centre Ville


In [62]:
#C.1.2 declared_income_2021 - Rename columns

In [63]:
rename_dict = {
    "IRIS": "iris_id",
    "DEC_PIMP21": "menages_fiscaux_imposes_pct",
    "DEC_TP6021": "taux_bas_revenus_pct",
    "DEC_Q121": "revenu_q1_eur",
    "DEC_MED21": "revenu_median_eur",
    "DEC_Q321": "revenu_q3_eur",
    "DEC_D121": "revenu_d1_eur",
    "DEC_D921": "revenu_d9_eur",
    "DEC_PACT21": "part_revenus_activite_pct",
    "DEC_PPEN21": "part_pensions_pct",
    "DEC_PAUT21": "part_autres_revenus_pct"
}
declared_income_2021.rename(
    columns=rename_dict,
    inplace=True
)
declared_income_2021.head()

,iris_id,menages_fiscaux_imposes_pct,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_autres_revenus_pct
0,010040101,"43,0","29,0",12610,19330,26390,7760,33850,"69,3","27,1","3,6"
1,010040102,"42,0","39,0",9730,16830,24620,4680,33030,"70,5","25,6","3,9"
2,010040201,"47,0","29,0",12220,19940,27650,6500,37220,"69,3","26,7","4,0"
3,010040202,"62,0","14,0",18350,25560,35010,11960,45520,"65,5","21,8","12,7"
4,010330102,"42,0","31,0",11280,19870,30050,5250,44820,"74,1","23,0","2,9"


In [64]:
#C.2.2 disposable_income_2021 - Rename columns

In [65]:
rename_dict = {
    "IRIS": "iris_id",
    "DISP_TP6021": "taux_bas_revenus_pct",
    "DISP_Q121": "revenu_q1_eur",
    "DISP_MED21": "revenu_median_eur",
    "DISP_Q321": "revenu_q3_eur",
    "DISP_D121": "revenu_d1_eur",
    "DISP_D921": "revenu_d9_eur",
    "DISP_PACT21": "part_revenus_activite_pct",
    "DISP_PPEN21": "part_pensions_pct",
    "DISP_PPAT21": "part_revenus_patrimoine_pct",
    "DISP_PPSOC21": "part_prestations_sociales_pct",
    "DISP_PPFAM21": "part_prestations_familiales_pct",
    "DISP_PPMINI21": "part_minima_sociaux_pct",
    "DISP_PPLOGT21": "part_prestations_logement_pct",
    "DISP_PIMPOT21": "part_impots_pct"
}
disposable_income_2021.rename(
    columns=rename_dict,
    inplace=True
)
disposable_income_2021.head()

,iris_id,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_revenus_patrimoine_pct,part_prestations_sociales_pct,part_prestations_familiales_pct,part_minima_sociaux_pct,part_prestations_logement_pct,part_impots_pct
0,010040101,"19,0",14990,20350,26140,11620,32060,"70,8","26,9","6,2","8,6","3,3","3,8","1,5","-12,5"
1,010040102,"25,0",13880,18570,24760,10580,31130,"70,6","24,9","5,8","11,1","3,7","5,1","2,3","-12,4"
2,010040201,"19,0",15190,20700,27180,11400,34450,"72,5","27,2","6,4","7,7","2,8","3,3","1,6","-13,8"
3,010040202,"8,0",19600,25230,33170,14810,41230,"73,3","23,8","16,2","4,0","1,8","1,5","0,7","-17,3"
4,010330102,"24,0",14050,20420,29640,9410,42390,"78,9","23,7","5,2","5,3","1,5","2,5","1,3","-13,1"


In [66]:
#C.3.2 insee_to_commune - Rename columns

In [67]:
rename_dict = {
    "COM": "insee_commune_id",
    "NCC": "commune_name_upper",
    "NCCENR": "commune_name"
}
insee_to_commune.rename(
    columns=rename_dict,
    inplace=True
)
insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes
4,01006,AMBLEON,Ambléon


In [68]:
#C.4.2 iris_to_commune - Rename columns

In [69]:
rename_dict = {
    "COD_MOD": "iris_id",
    "LIB_MOD": "place_in_commune"
}
iris_to_commune.rename(
    columns=rename_dict,
    inplace=True
)
iris_to_commune.head()

,iris_id,place_in_commune
0,010040101,Les Pérouses-Triangle d'Activités
1,010040102,Longeray-Gare
2,010040201,Centre-Saint-Germain-Vareilles
3,010040202,Tiret-Les Allymes
4,010330102,Centre Ville


In [70]:
#C.4.1.3 declared_income_2021 - CAST numeric data appearing as STR 

In [71]:
declared_income_2021.dtypes

iris_id                        str
menages_fiscaux_imposes_pct    str
taux_bas_revenus_pct           str
revenu_q1_eur                  str
revenu_median_eur              str
revenu_q3_eur                  str
revenu_d1_eur                  str
revenu_d9_eur                  str
part_revenus_activite_pct      str
part_pensions_pct              str
part_autres_revenus_pct        str
dtype: object

In [72]:
for col in declared_income_2021.columns:
    if col != "iris_id":
        declared_income_2021[col] = pd.to_numeric(declared_income_2021[col].str.replace(",", ".", regex=False), errors="coerce") 
        # regex=False : Replace this character exactly, without using a regular expression
        # errors="coerce" ==> when conversion fails, change value in NaN
declared_income_2021.head()

,iris_id,menages_fiscaux_imposes_pct,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_autres_revenus_pct
0,010040101,43.0,29.0,12610.0,19330.0,26390.0,7760.0,33850.0,69.3,27.1,3.6
1,010040102,42.0,39.0,9730.0,16830.0,24620.0,4680.0,33030.0,70.5,25.6,3.9
2,010040201,47.0,29.0,12220.0,19940.0,27650.0,6500.0,37220.0,69.3,26.7,4.0
3,010040202,62.0,14.0,18350.0,25560.0,35010.0,11960.0,45520.0,65.5,21.8,12.7
4,010330102,42.0,31.0,11280.0,19870.0,30050.0,5250.0,44820.0,74.1,23.0,2.9


In [73]:
declared_income_2021.dtypes

iris_id                            str
menages_fiscaux_imposes_pct    float64
taux_bas_revenus_pct           float64
revenu_q1_eur                  float64
revenu_median_eur              float64
revenu_q3_eur                  float64
revenu_d1_eur                  float64
revenu_d9_eur                  float64
part_revenus_activite_pct      float64
part_pensions_pct              float64
part_autres_revenus_pct        float64
dtype: object

In [74]:
#C.4.2.3 disposable_income_2021 - CAST numeric data appearing as STR 

In [75]:
for col in disposable_income_2021.columns:
    if col != "iris_id":
        disposable_income_2021[col] = pd.to_numeric(disposable_income_2021[col].str.replace(",", ".", regex=False), errors="coerce")
disposable_income_2021.dtypes

iris_id                                str
taux_bas_revenus_pct               float64
revenu_q1_eur                      float64
revenu_median_eur                  float64
revenu_q3_eur                      float64
revenu_d1_eur                      float64
revenu_d9_eur                      float64
part_revenus_activite_pct          float64
part_pensions_pct                  float64
part_revenus_patrimoine_pct        float64
part_prestations_sociales_pct      float64
part_prestations_familiales_pct    float64
part_minima_sociaux_pct            float64
part_prestations_logement_pct      float64
part_impots_pct                    float64
dtype: object

In [76]:
#C.4.1.4 declared_income_2021 - Check if NaN values

In [77]:
declared_income_2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 16026 entries, 0 to 16025
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   iris_id                      16026 non-null  str    
 1   menages_fiscaux_imposes_pct  14490 non-null  float64
 2   taux_bas_revenus_pct         14489 non-null  float64
 3   revenu_q1_eur                14490 non-null  float64
 4   revenu_median_eur            14490 non-null  float64
 5   revenu_q3_eur                14490 non-null  float64
 6   revenu_d1_eur                14490 non-null  float64
 7   revenu_d9_eur                14490 non-null  float64
 8   part_revenus_activite_pct    14490 non-null  float64
 9   part_pensions_pct            14490 non-null  float64
 10  part_autres_revenus_pct      14490 non-null  float64
dtypes: float64(10), str(1)
memory usage: 1.3 MB


In [78]:
# OBSERVATION: declared_income_2021 - After conversion from String to Float, some values appear as NaN

In [79]:
#Select empty lines (appart from IRIS code)
cols = [col for col in declared_income_2021.columns if col != "iris_id"]
null_rows_declared = declared_income_2021[declared_income_2021[cols].isna().all(axis=1)]
null_rows_declared.head(30)

,iris_id,menages_fiscaux_imposes_pct,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_autres_revenus_pct
59,012830302,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63,012830306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,021680301,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,024080102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92,024080103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,024080701,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,026910210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
127,026910505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
176,031850107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
178,031850109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [80]:
#C.4.2.4 disposable_income_2021 - Check if NaN values

In [81]:
disposable_income_2021.info()

<class 'pandas.DataFrame'>
RangeIndex: 16026 entries, 0 to 16025
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   iris_id                          16026 non-null  str    
 1   taux_bas_revenus_pct             14486 non-null  float64
 2   revenu_q1_eur                    14490 non-null  float64
 3   revenu_median_eur                14490 non-null  float64
 4   revenu_q3_eur                    14490 non-null  float64
 5   revenu_d1_eur                    14490 non-null  float64
 6   revenu_d9_eur                    14490 non-null  float64
 7   part_revenus_activite_pct        14490 non-null  float64
 8   part_pensions_pct                14490 non-null  float64
 9   part_revenus_patrimoine_pct      14490 non-null  float64
 10  part_prestations_sociales_pct    14490 non-null  float64
 11  part_prestations_familiales_pct  14490 non-null  float64
 12  part_minima_sociaux_pct      

In [82]:
# OBSERVATION: disposable_income_2021 - After conversion from String to Float, some values appear as NaN

In [83]:
#Select empty lines (appart from IRIS code)
cols = [col for col in disposable_income_2021.columns if col != "iris_id"]
null_rows_disposable = disposable_income_2021[disposable_income_2021[cols].isna().all(axis=1)]
null_rows_disposable.head(30)

,iris_id,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_revenus_patrimoine_pct,part_prestations_sociales_pct,part_prestations_familiales_pct,part_minima_sociaux_pct,part_prestations_logement_pct,part_impots_pct
59,012830302,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
63,012830306,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,021680301,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
91,024080102,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
92,024080103,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,024080701,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,026910210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
127,026910505,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
176,031850107,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
178,031850109,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
null_rows_declared.info()

<class 'pandas.DataFrame'>
Index: 1536 entries, 59 to 16025
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   iris_id                      1536 non-null   str    
 1   menages_fiscaux_imposes_pct  0 non-null      float64
 2   taux_bas_revenus_pct         0 non-null      float64
 3   revenu_q1_eur                0 non-null      float64
 4   revenu_median_eur            0 non-null      float64
 5   revenu_q3_eur                0 non-null      float64
 6   revenu_d1_eur                0 non-null      float64
 7   revenu_d9_eur                0 non-null      float64
 8   part_revenus_activite_pct    0 non-null      float64
 9   part_pensions_pct            0 non-null      float64
 10  part_autres_revenus_pct      0 non-null      float64
dtypes: float64(10), str(1)
memory usage: 144.0 KB


In [85]:
null_rows_disposable.info()

<class 'pandas.DataFrame'>
Index: 1536 entries, 59 to 16025
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   iris_id                          1536 non-null   str    
 1   taux_bas_revenus_pct             0 non-null      float64
 2   revenu_q1_eur                    0 non-null      float64
 3   revenu_median_eur                0 non-null      float64
 4   revenu_q3_eur                    0 non-null      float64
 5   revenu_d1_eur                    0 non-null      float64
 6   revenu_d9_eur                    0 non-null      float64
 7   part_revenus_activite_pct        0 non-null      float64
 8   part_pensions_pct                0 non-null      float64
 9   part_revenus_patrimoine_pct      0 non-null      float64
 10  part_prestations_sociales_pct    0 non-null      float64
 11  part_prestations_familiales_pct  0 non-null      float64
 12  part_minima_sociaux_pct          0

In [86]:
# OBSERVATION: in both revenue datasets, rows with missing data are completely empty except for the location code. There are no partially filled rows. 
# This indicates that some locations have no available revenue data.
# Additionally, the number of such rows is identical across both datasets, which supports data consistency and reliability. 
# Both datasets reflect the same underlying coverage gaps, despite representing slightly different analytical perspectives.

In [87]:
#C.4.3.4 insee_to_commune - Check if NaN values

In [88]:
insee_to_commune.info()

<class 'pandas.DataFrame'>
Index: 34875 entries, 0 to 37547
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   insee_commune_id    34875 non-null  str  
 1   commune_name_upper  34875 non-null  str  
 2   commune_name        34875 non-null  str  
dtypes: str(3)
memory usage: 1.1 MB


In [89]:
# OBSERVATION: No NULL values in insee_to_commune table

In [90]:
#C.4.4.4 iris_to_commune - Check if NaN values

In [91]:
iris_to_commune.info()

<class 'pandas.DataFrame'>
RangeIndex: 16026 entries, 0 to 16025
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   iris_id           16026 non-null  str  
 1   place_in_commune  16026 non-null  str  
dtypes: str(2)
memory usage: 250.5 KB


In [92]:
# OBSERVATION: No NULL values in iris_to_commune table
# Additionally, all three datasets—iris_to_commune, declared_income_2021, and disposable_income_2021—contain the same number of entries, suggesting full alignment across datasets.

In [93]:
# DECISION: rows with missing values in the revenue datasets will be removed for analysis purposes.
# However, a flag will be added to the iris_to_commune table to track locations with missing revenue data, as they represent valid geographic entities with no available information.
# Removing these rows without preserving this information would introduce bias by excluding areas with missing data.

In [94]:
#C.4.1.5 declared_income_2021 - Remove empty lines

In [95]:
declared_income_2021 = declared_income_2021.drop(index=null_rows_declared.index)

In [96]:
declared_income_2021.info()

<class 'pandas.DataFrame'>
Index: 14490 entries, 0 to 16022
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   iris_id                      14490 non-null  str    
 1   menages_fiscaux_imposes_pct  14490 non-null  float64
 2   taux_bas_revenus_pct         14489 non-null  float64
 3   revenu_q1_eur                14490 non-null  float64
 4   revenu_median_eur            14490 non-null  float64
 5   revenu_q3_eur                14490 non-null  float64
 6   revenu_d1_eur                14490 non-null  float64
 7   revenu_d9_eur                14490 non-null  float64
 8   part_revenus_activite_pct    14490 non-null  float64
 9   part_pensions_pct            14490 non-null  float64
 10  part_autres_revenus_pct      14490 non-null  float64
dtypes: float64(10), str(1)
memory usage: 1.3 MB


In [97]:
#C.4.2.5 disposable_income_2021 - Remove empty lines

In [98]:
disposable_income_2021 = disposable_income_2021.drop(index=null_rows_disposable.index)

In [99]:
disposable_income_2021.info()

<class 'pandas.DataFrame'>
Index: 14490 entries, 0 to 16022
Data columns (total 15 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   iris_id                          14490 non-null  str    
 1   taux_bas_revenus_pct             14486 non-null  float64
 2   revenu_q1_eur                    14490 non-null  float64
 3   revenu_median_eur                14490 non-null  float64
 4   revenu_q3_eur                    14490 non-null  float64
 5   revenu_d1_eur                    14490 non-null  float64
 6   revenu_d9_eur                    14490 non-null  float64
 7   part_revenus_activite_pct        14490 non-null  float64
 8   part_pensions_pct                14490 non-null  float64
 9   part_revenus_patrimoine_pct      14490 non-null  float64
 10  part_prestations_sociales_pct    14490 non-null  float64
 11  part_prestations_familiales_pct  14490 non-null  float64
 12  part_minima_sociaux_pct          1

In [100]:
#C.4.4.5 iris_to_commune - Flag communes with empty revenu data in iris_to_commune table

In [101]:
iris_with_data = set(declared_income_2021["iris_id"])
iris_to_commune["has_income_data"] = iris_to_commune["iris_id"].isin(iris_with_data)
iris_to_commune.head()

,iris_id,place_in_commune,has_income_data
0,010040101,Les Pérouses-Triangle d'Activités,True
1,010040102,Longeray-Gare,True
2,010040201,Centre-Saint-Germain-Vareilles,True
3,010040202,Tiret-Les Allymes,True
4,010330102,Centre Ville,True


In [102]:
# DECISION: enhance the iris_to_commune reference table by adding municipality-level information (INSEE code and commune name).
# Reminder: the IRIS level corresponds to a sub-municipal geographic unit within a city.

In [103]:
#C.4.4.6 iris_to_commune - add a column insee_commune_id and fill it with the first 5 chars of IRIS CODE

In [104]:
pos = iris_to_commune.columns.get_loc("iris_id")
iris_to_commune.insert(loc=pos + 1,column="insee_commune_id",value=iris_to_commune["iris_id"].str[:5])

In [105]:
iris_to_commune.head()

,iris_id,insee_commune_id,place_in_commune,has_income_data
0,010040101,01004,Les Pérouses-Triangle d'Activités,True
1,010040102,01004,Longeray-Gare,True
2,010040201,01004,Centre-Saint-Germain-Vareilles,True
3,010040202,01004,Tiret-Les Allymes,True
4,010330102,01033,Centre Ville,True


In [106]:
pos = iris_to_commune.columns.get_loc("insee_commune_id")
iris_to_commune.insert(
             loc=pos + 1,
             column="commune_name",
             value=duckdb.query("""
                SELECT 
                    c.commune_name
                    FROM iris_to_commune i
                    LEFT JOIN (
                        SELECT insee_commune_id, commune_name
                        FROM insee_to_commune
                    ) c
                    ON i.insee_commune_id = c.insee_commune_id
                    ORDER BY i.iris_id
                """).df()
            )

In [107]:
iris_to_commune.head(30)

,iris_id,insee_commune_id,commune_name,place_in_commune,has_income_data
0,010040101,01004,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,True
1,010040102,01004,Ambérieu-en-Bugey,Longeray-Gare,True
2,010040201,01004,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,True
3,010040202,01004,Ambérieu-en-Bugey,Tiret-Les Allymes,True
4,010330102,01033,Valserhône,Centre Ville,True
5,010330103,01033,Valserhône,Lancrans-Coupy-Vanchy,True
6,010330201,01033,Valserhône,Arc Vouvray-Gare-Châtillon,True
7,010330202,01033,Valserhône,Plateau de Musinens,True
8,010330301,01033,Valserhône,Arlod,True
9,010330401,01033,Valserhône,Châtillon-en-Michaille,True


In [108]:
#D - Save all cleaned tables in csv formats in cleaned_data folder

In [111]:
declared_income_2021.to_csv("cleaned_data/declared_income_2021_clean.csv", index=False)
disposable_income_2021.to_csv("cleaned_data/disposable_income_2021_clean.csv", index=False)
insee_to_commune.to_csv("cleaned_data/insee_to_commune_clean.csv", index=False)
iris_to_commune.to_csv("cleaned_data/iris_to_commune_clean.csv", index=False)

In [112]:
#======================================================
# II - SECURITY / SECURITE
#======================================================

In [113]:
#------------------------------------------------------
# II.a - CRIME analysis / Analyse de DELINQUANCE
#------------------------------------------------------

In [155]:
# A - Download compressed file 

In [115]:
url3 = "https://www.data.gouv.fr/api/1/datasets/r/6252a84c-6b9e-4415-a743-fc6a631877bb"
download(url3, "source_files/Security/donnee-data.gouv-2024-geographie2025-produit-le2025-06-04.csv.gz")

Download successful: source_files/Security/donnee-data.gouv-2024-geographie2025-produit-le2025-06-04.csv.gz


In [116]:
# B - Read dataset 

In [117]:
crime = pd.read_csv("source_files/Security/donnee-data.gouv-2024-geographie2025-produit-le2025-06-04.csv.gz", sep=";", compression="gzip", dtype={"CODGEO_2025": str})
crime.head()

,CODGEO_2025,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,est_diffuse,insee_pop,insee_pop_millesime,insee_log,insee_log_millesime,complement_info_nombre,complement_info_taux
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,"0,0000000",diff,767,2016,348,2016,NaN,NaN
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,ndiff,767,2016,348,2016,"1,3620690","0,9598386"
2,01001,2016,Violences sexuelles,Victime,0.0,"0,0000000",diff,767,2016,348,2016,NaN,NaN
3,01001,2016,Vols avec armes,Infraction,0.0,"0,0000000",diff,767,2016,348,2016,NaN,NaN
4,01001,2016,Vols violents sans arme,Infraction,0.0,"0,0000000",diff,767,2016,348,2016,NaN,NaN


In [118]:
crime.info()

<class 'pandas.DataFrame'>
RangeIndex: 4714200 entries, 0 to 4714199
Data columns (total 13 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   CODGEO_2025             str    
 1   annee                   int64  
 2   indicateur              str    
 3   unite_de_compte         str    
 4   nombre                  float64
 5   taux_pour_mille         str    
 6   est_diffuse             str    
 7   insee_pop               int64  
 8   insee_pop_millesime     int64  
 9   insee_log               int64  
 10  insee_log_millesime     int64  
 11  complement_info_nombre  str    
 12  complement_info_taux    str    
dtypes: float64(1), int64(5), str(7)
memory usage: 467.6 MB


In [119]:
# OBSERVATION: the dataset is large and should be considered when designing future processing steps.
# Data types appear to be consistent except taux_pour_mille which is a numaric value appearing as string.

In [120]:
# C - Clean dataset 

In [121]:
#C.1 - crime - Keep only needed columns

In [122]:
cols_to_drop = [
    "est_diffuse",
    "insee_pop_millesime",
    "insee_log",
    "insee_log_millesime",
    "complement_info_nombre",
    "complement_info_taux"
]
crime.drop(
    columns=cols_to_drop,
    inplace=True
)
crime.head()

,CODGEO_2025,annee,indicateur,unite_de_compte,nombre,taux_pour_mille,insee_pop
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,"0,0000000",767
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,767
2,01001,2016,Violences sexuelles,Victime,0.0,"0,0000000",767
3,01001,2016,Vols avec armes,Infraction,0.0,"0,0000000",767
4,01001,2016,Vols violents sans arme,Infraction,0.0,"0,0000000",767


In [123]:
#C.2 - crime - Rename columns

In [124]:
rename_dict = {
    "CODGEO_2025":"insee_commune_id",
    "indicateur":"crime_type",
    "insee_pop":"insee_pop_ref"
}
crime.rename(
    columns=rename_dict,
    inplace=True
)
crime.head()

,insee_commune_id,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,"0,0000000",767
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,767
2,01001,2016,Violences sexuelles,Victime,0.0,"0,0000000",767
3,01001,2016,Vols avec armes,Infraction,0.0,"0,0000000",767
4,01001,2016,Vols violents sans arme,Infraction,0.0,"0,0000000",767


In [125]:
#C.3 - crime - Perform needed cast - 

In [126]:
crime.dtypes

insee_commune_id        str
annee                 int64
crime_type              str
unite_de_compte         str
nombre              float64
taux_pour_mille         str
insee_pop_ref         int64
dtype: object

In [127]:
crime["taux_pour_mille"] = pd.to_numeric(crime["taux_pour_mille"].str.replace(",", ".", regex=False), errors="coerce")
        # errors="coerce" ==> when conversion fails, change value in NaN
crime.head()

,insee_commune_id,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,0.0,767
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,767
2,01001,2016,Violences sexuelles,Victime,0.0,0.0,767
3,01001,2016,Vols avec armes,Infraction,0.0,0.0,767
4,01001,2016,Vols violents sans arme,Infraction,0.0,0.0,767


In [128]:
crime.dtypes

insee_commune_id        str
annee                 int64
crime_type              str
unite_de_compte         str
nombre              float64
taux_pour_mille     float64
insee_pop_ref         int64
dtype: object

In [129]:
#C.4 - crime - Check null values

In [130]:
crime.info()

<class 'pandas.DataFrame'>
RangeIndex: 4714200 entries, 0 to 4714199
Data columns (total 7 columns):
 #   Column            Dtype  
---  ------            -----  
 0   insee_commune_id  str    
 1   annee             int64  
 2   crime_type        str    
 3   unite_de_compte   str    
 4   nombre            float64
 5   taux_pour_mille   float64
 6   insee_pop_ref     int64  
dtypes: float64(2), int64(2), str(3)
memory usage: 251.8 MB


In [131]:
# The dataset is too large for DataFrame.info() to display non-null value counts.

In [132]:
crime.isna().sum()

insee_commune_id          0
annee                     0
crime_type                0
unite_de_compte           0
nombre              2174414
taux_pour_mille     2175177
insee_pop_ref             0
dtype: int64

In [133]:
# DECISION: add a data_available column to enable future filtering based on data availability, without removing rows in order to prevent bias.

In [134]:
crime["data_available"] = ~crime["nombre"].isna()

In [135]:
crime.head()

,insee_commune_id,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref,data_available
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,0.0,767,True
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,767,False
2,01001,2016,Violences sexuelles,Victime,0.0,0.0,767,True
3,01001,2016,Vols avec armes,Infraction,0.0,0.0,767,True
4,01001,2016,Vols violents sans arme,Infraction,0.0,0.0,767,True


In [136]:
#count non zero and non null number of crimes

In [137]:
((crime["nombre"] != 0) & (crime["nombre"].notna())).sum()

np.int64(387422)

In [138]:
#create another table with only the non zero, non null crime occurences

In [139]:
known_crimes = crime[crime["nombre"] > 0].copy()
known_crimes.info()

<class 'pandas.DataFrame'>
Index: 387422 entries, 30 to 4714199
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   insee_commune_id  387422 non-null  str    
 1   annee             387422 non-null  int64  
 2   crime_type        387422 non-null  str    
 3   unite_de_compte   387422 non-null  str    
 4   nombre            387422 non-null  float64
 5   taux_pour_mille   387422 non-null  float64
 6   insee_pop_ref     387422 non-null  int64  
 7   data_available    387422 non-null  bool   
dtypes: bool(1), float64(2), int64(2), str(3)
memory usage: 24.0 MB


In [140]:
# DECISION : use the insee_to_commune table as a reference index to track data availability across datasets (e.g., crime and revenue data).

In [141]:
#C.5 - known_crimes & insee_to_commune - Flag communes with empty crime data in insee_to_commune table

In [142]:
insee_with_data = set(known_crimes["insee_commune_id"])
insee_to_commune["has_crime_data"] = insee_to_commune["insee_commune_id"].isin(insee_with_data)
insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True
4,01006,AMBLEON,Ambléon,False


In [143]:
# Add revenue data availability indicators to the insee_to_commune table
# For each commune, compute:
# - whether revenue data is partially available
# - whether revenue data is fully available
# - the revenue data coverage rate

In [144]:
income_coverage = (
    iris_to_commune
    .groupby("insee_commune_id")["has_income_data"]
    .agg(
        has_income_data_any="any",
        has_income_data_all="all",
        income_data_coverage="mean"
    )
    .reset_index()
)
income_coverage.head(30)

,insee_commune_id,has_income_data_any,has_income_data_all,income_data_coverage
0,01004,True,True,1.000000
1,01033,True,True,1.000000
2,01034,True,True,1.000000
3,01053,True,True,1.000000
4,01143,True,True,1.000000
5,01160,True,True,1.000000
6,01173,True,True,1.000000
7,01194,True,True,1.000000
8,01202,True,True,1.000000
9,01244,True,True,1.000000


In [145]:
insee_to_commune = insee_to_commune.merge(
    income_coverage,
    on="insee_commune_id",
    how="left"
)

insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False,NaN,NaN,NaN
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False,NaN,NaN,NaN
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True,True,True,1.0
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True,NaN,NaN,NaN
4,01006,AMBLEON,Ambléon,False,NaN,NaN,NaN


In [146]:
# Replace NaN values (due to left join) with meaningful defaults (e.g., False or 0) to reflect data availability

In [147]:
insee_to_commune["has_income_data_any"] = (
    insee_to_commune["has_income_data_any"]
    .fillna(False)
    .astype(bool)
)

insee_to_commune["has_income_data_all"] = (
    insee_to_commune["has_income_data_all"]
    .fillna(False)
    .astype(bool)
)

insee_to_commune["income_data_coverage"] = (
    insee_to_commune["income_data_coverage"]
    .fillna(0)
)

In [148]:
insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False,False,False,0.0
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False,False,False,0.0
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True,True,True,1.0
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True,False,False,0.0
4,01006,AMBLEON,Ambléon,False,False,False,0.0


In [149]:
#C.6 Join commune_name into known_crimes table

In [150]:
known_crimes.head()

,insee_commune_id,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref,data_available
30,01004,2016,Violences physiques intrafamiliales,Victime,28.0,1.988495,14081,True
31,01004,2016,Violences physiques hors cadre familial,Victime,53.0,3.763937,14081,True
32,01004,2016,Violences sexuelles,Victime,13.0,0.923230,14081,True
35,01004,2016,Vols sans violence contre des personnes,Victime entendue,133.0,9.445352,14081,True
36,01004,2016,Cambriolages de logement,Infraction,72.0,10.103681,14081,True


In [151]:
commune_map = (
    insee_to_commune
    .drop_duplicates(subset="insee_commune_id")
    .set_index("insee_commune_id")["commune_name"]
)

pos = known_crimes.columns.get_loc("insee_commune_id")

known_crimes.insert(
    loc=pos + 1,
    column="commune_name",
    value=known_crimes["insee_commune_id"].map(commune_map)
)

known_crimes.head(30)

,insee_commune_id,commune_name,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref,data_available
30,01004,Ambérieu-en-Bugey,2016,Violences physiques intrafamiliales,Victime,28.0,1.988495,14081,True
31,01004,Ambérieu-en-Bugey,2016,Violences physiques hors cadre familial,Victime,53.0,3.763937,14081,True
32,01004,Ambérieu-en-Bugey,2016,Violences sexuelles,Victime,13.0,0.923230,14081,True
35,01004,Ambérieu-en-Bugey,2016,Vols sans violence contre des personnes,Victime entendue,133.0,9.445352,14081,True
36,01004,Ambérieu-en-Bugey,2016,Cambriolages de logement,Infraction,72.0,10.103681,14081,True
37,01004,Ambérieu-en-Bugey,2016,Vols de véhicule,Véhicule,49.0,3.479866,14081,True
38,01004,Ambérieu-en-Bugey,2016,Vols dans les véhicules,Véhicule,46.0,3.266813,14081,True
39,01004,Ambérieu-en-Bugey,2016,Vols d'accessoires sur véhicules,Véhicule,22.0,1.562389,14081,True
40,01004,Ambérieu-en-Bugey,2016,Destructions et dégradations volontaires,Infraction,116.0,8.238051,14081,True
41,01004,Ambérieu-en-Bugey,2016,Usage de stupéfiants,Mis en cause,29.0,2.059513,14081,True


In [152]:
# add commune_name and place_in_commune name into disposable_income_2021 table

In [153]:
iris_to_commune.head()

,iris_id,insee_commune_id,commune_name,place_in_commune,has_income_data
0,010040101,01004,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,True
1,010040102,01004,Ambérieu-en-Bugey,Longeray-Gare,True
2,010040201,01004,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,True
3,010040202,01004,Ambérieu-en-Bugey,Tiret-Les Allymes,True
4,010330102,01033,Valserhône,Centre Ville,True


In [154]:
disposable_income_2021.head()

,iris_id,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_revenus_patrimoine_pct,part_prestations_sociales_pct,part_prestations_familiales_pct,part_minima_sociaux_pct,part_prestations_logement_pct,part_impots_pct
0,010040101,19.0,14990.0,20350.0,26140.0,11620.0,32060.0,70.8,26.9,6.2,8.6,3.3,3.8,1.5,-12.5
1,010040102,25.0,13880.0,18570.0,24760.0,10580.0,31130.0,70.6,24.9,5.8,11.1,3.7,5.1,2.3,-12.4
2,010040201,19.0,15190.0,20700.0,27180.0,11400.0,34450.0,72.5,27.2,6.4,7.7,2.8,3.3,1.6,-13.8
3,010040202,8.0,19600.0,25230.0,33170.0,14810.0,41230.0,73.3,23.8,16.2,4.0,1.8,1.5,0.7,-17.3
4,010330102,24.0,14050.0,20420.0,29640.0,9410.0,42390.0,78.9,23.7,5.2,5.3,1.5,2.5,1.3,-13.1


In [155]:
iris_map = (
    iris_to_commune
    .drop_duplicates(subset="iris_id")
    .set_index("iris_id")[["commune_name", "place_in_commune"]]
)

# Add the columns first
disposable_income_2021 = disposable_income_2021.join(
    iris_map,
    on="iris_id"
)

In [156]:
# Then move them
disposable_income_2021 = move_columns(
    disposable_income_2021,
    ["commune_name", "place_in_commune"],
    after="iris_id"
)

disposable_income_2021.head(30)

,iris_id,commune_name,place_in_commune,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_revenus_patrimoine_pct,part_prestations_sociales_pct,part_prestations_familiales_pct,part_minima_sociaux_pct,part_prestations_logement_pct,part_impots_pct
0,010040101,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,19.0,14990.0,20350.0,26140.0,11620.0,32060.0,70.8,26.9,6.2,8.6,3.3,3.8,1.5,-12.5
1,010040102,Ambérieu-en-Bugey,Longeray-Gare,25.0,13880.0,18570.0,24760.0,10580.0,31130.0,70.6,24.9,5.8,11.1,3.7,5.1,2.3,-12.4
2,010040201,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,19.0,15190.0,20700.0,27180.0,11400.0,34450.0,72.5,27.2,6.4,7.7,2.8,3.3,1.6,-13.8
3,010040202,Ambérieu-en-Bugey,Tiret-Les Allymes,8.0,19600.0,25230.0,33170.0,14810.0,41230.0,73.3,23.8,16.2,4.0,1.8,1.5,0.7,-17.3
4,010330102,Valserhône,Centre Ville,24.0,14050.0,20420.0,29640.0,9410.0,42390.0,78.9,23.7,5.2,5.3,1.5,2.5,1.3,-13.1
5,010330103,Valserhône,Lancrans-Coupy-Vanchy,17.0,16270.0,24060.0,35760.0,10700.0,49310.0,82.7,21.8,5.8,3.9,1.5,1.5,0.9,-14.2
6,010330201,Valserhône,Arc Vouvray-Gare-Châtillon,15.0,17940.0,25980.0,36690.0,11760.0,53670.0,79.2,26.4,6.0,3.2,1.2,1.3,0.7,-14.8
7,010330202,Valserhône,Plateau de Musinens,26.0,13750.0,19630.0,28050.0,10290.0,40430.0,70.2,30.4,3.7,8.0,2.6,3.7,1.7,-12.3
8,010330301,Valserhône,Arlod,19.0,15590.0,22360.0,31880.0,11260.0,43590.0,83.8,19.1,3.0,5.8,2.8,1.9,1.1,-11.7
9,010330401,Valserhône,Châtillon-en-Michaille,8.0,20830.0,28590.0,39750.0,15290.0,54750.0,86.2,19.2,7.3,2.6,1.3,0.8,0.5,-15.3


In [157]:
# add commune_name and place_in_commune name into declared_income_2021 table

In [158]:
# Add the columns first
declared_income_2021 = declared_income_2021.join(
    iris_map,
    on="iris_id"
)
# Then move them
declared_income_2021 = move_columns(
    declared_income_2021,
    ["commune_name", "place_in_commune"],
    after="iris_id"
)

declared_income_2021.head(30)

,iris_id,commune_name,place_in_commune,menages_fiscaux_imposes_pct,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_autres_revenus_pct
0,010040101,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,43.0,29.0,12610.0,19330.0,26390.0,7760.0,33850.0,69.3,27.1,3.6
1,010040102,Ambérieu-en-Bugey,Longeray-Gare,42.0,39.0,9730.0,16830.0,24620.0,4680.0,33030.0,70.5,25.6,3.9
2,010040201,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,47.0,29.0,12220.0,19940.0,27650.0,6500.0,37220.0,69.3,26.7,4.0
3,010040202,Ambérieu-en-Bugey,Tiret-Les Allymes,62.0,14.0,18350.0,25560.0,35010.0,11960.0,45520.0,65.5,21.8,12.7
4,010330102,Valserhône,Centre Ville,42.0,31.0,11280.0,19870.0,30050.0,5250.0,44820.0,74.1,23.0,2.9
5,010330103,Valserhône,Lancrans-Coupy-Vanchy,49.0,23.0,14480.0,24250.0,37150.0,7390.0,52060.0,76.1,20.6,3.3
6,010330201,Valserhône,Arc Vouvray-Gare-Châtillon,50.0,19.0,16680.0,26260.0,38330.0,8940.0,57920.0,71.9,24.6,3.5
7,010330202,Valserhône,Plateau de Musinens,40.0,36.0,10810.0,18380.0,28270.0,5630.0,42550.0,68.1,30.3,1.6
8,010330301,Valserhône,Arlod,39.0,27.0,13000.0,21760.0,32830.0,7590.0,45980.0,79.8,18.7,1.5
9,010330401,Valserhône,Châtillon-en-Michaille,53.0,12.0,20080.0,29300.0,41880.0,13120.0,58590.0,77.5,17.6,4.9


In [159]:
# Remove data_available column as all rows have data

In [160]:
known_crimes.drop(
    columns="data_available",
    inplace=True
)
known_crimes.head()

,insee_commune_id,commune_name,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref
30,01004,Ambérieu-en-Bugey,2016,Violences physiques intrafamiliales,Victime,28.0,1.988495,14081
31,01004,Ambérieu-en-Bugey,2016,Violences physiques hors cadre familial,Victime,53.0,3.763937,14081
32,01004,Ambérieu-en-Bugey,2016,Violences sexuelles,Victime,13.0,0.923230,14081
35,01004,Ambérieu-en-Bugey,2016,Vols sans violence contre des personnes,Victime entendue,133.0,9.445352,14081
36,01004,Ambérieu-en-Bugey,2016,Cambriolages de logement,Infraction,72.0,10.103681,14081


In [161]:
#D - Save all cleaned tables in csv formats in cleaned_data folder

In [162]:
known_crimes.to_csv("cleaned_data/known_crimes_clean.csv", index=False)
crime.to_csv("cleaned_data/crime_clean.csv", index=False)
declared_income_2021.to_csv("cleaned_data/declared_income_2021_clean.csv", index=False)
disposable_income_2021.to_csv("cleaned_data/disposable_income_2021_clean.csv", index=False)
insee_to_commune.to_csv("cleaned_data/insee_to_commune_clean.csv", index=False)
iris_to_commune.to_csv("cleaned_data/iris_to_commune_clean.csv", index=False)

In [163]:
#--------------------------------------------------------------
# II.b - CRIME analysis / Analyse d'EFFECTIF POLICE MUNICIPALE
#--------------------------------------------------------------

In [164]:
# REALISATION ULTERIEURE

In [165]:
#sources: https://www.data.gouv.fr/datasets/police-municipale-effectifs-par-commune

In [166]:
#=========================================================================
# III - POLLUTION
#=========================================================================

In [167]:
#--------------------------------------------------------------------------------
# III.a - Quality of air analysis / Analyse de la qualite de l'air - source ATMO
#-------------------------------------------------------------------------------

In [168]:
# A - Download dataset - First login to ATMO France with private hidden data

In [169]:
login_url = f"{ATMO_FRANCE_BASE_URL}/api/login"

login_payload = {
    "username": ATMO_FRANCE_USERNAME,
    "password": ATMO_FRANCE_PASSWORD
}

token = None

try:
    response = requests.post(login_url, json=login_payload, timeout=30)
    response.raise_for_status()

    login_data = response.json()
    token = login_data.get("token")

    if not token:
        raise ValueError("Token non trouvé dans la réponse API")

    print("Authentification réussie ✔️")

except requests.exceptions.RequestException as e:
    print(f"Erreur API : {e}")

except ValueError as val_err:
    print(f"Erreur logique : {val_err}")

if (token != None):
    print("Token not None")

Authentification réussie ✔️
Token not None


In [170]:
# B - Download dataset - Extract ATMO indices at city/EPCI level

In [171]:
#=====================================================================================
# Fonction to extract ATMO indices (Air quality) via ATMO France API requests
#=====================================================================================
def get_atmo_indices_df(
    headers,
    base_url,
    aasqa,
    date_start,
    date_end,
    output_format="geojson"
):
    """
    Get back ATMO France indices for provided AASQA (group of cities) and provided period.
    Sends back pandas DataFrame.
    """
    indices_url = f"{base_url}/api/v2/data/indices/atmo"

    params = {
        "format": output_format,
        "date_historique": date_start,
        "date": date_end,
        "aasqa": aasqa
    }

    response = requests.get(indices_url, headers=headers, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    features = data.get("features", [])
    records = [f.get("properties", {}) for f in features]

    df = pd.DataFrame(records)
    return df

In [172]:
#=====================================================================================
# Fonction which translates year and quarter into start date and end date
#
# We retrieve quarter by quarter to prevent time out issues in ATMO server.
#=====================================================================================
def quarter_to_dates(year, quarter):
    """
    Translates year + quarter in begin-date --> end-date
    """
    quarter_map = {
        1: (f"{year}-01-01", f"{year}-03-31"),
        2: (f"{year}-04-01", f"{year}-06-30"),
        3: (f"{year}-07-01", f"{year}-09-30"),
        4: (f"{year}-10-01", f"{year}-12-31"),
    }

    if quarter not in quarter_map:
        raise ValueError("quarter doit être 1, 2, 3 ou 4")

    return quarter_map[quarter]

In [173]:
#===================================================================================================
# Extract ATMO indices by iterating over years, quarters, and AASQA zones
# Scope: Paris, Lyon, and Marseille (including suburbs) for an initial proof of concept
#===================================================================================================

headers = {
    "Authorization": f"Bearer {token}"
}

city_aasqa = {
    "Paris": "11",
    "Lyon": "84",
    "Marseille": "93"
}

years = [2024, 2025]
quarters = [1, 2, 3, 4]

all_dfs = []

for year in years:
    for quarter in quarters:
        date_start, date_end = quarter_to_dates(year, quarter)

        for city, aasqa in city_aasqa.items():
            try:
                df_city = get_atmo_indices_df(
                    headers=headers,
                    base_url=ATMO_FRANCE_BASE_URL,
                    aasqa=aasqa,
                    date_start=date_start,
                    date_end=date_end,
                    output_format="geojson"
                )

                df_city["city_target"] = city
                df_city["aasqa_target"] = aasqa
                df_city["year"] = year
                df_city["quarter"] = quarter
                df_city["date_start"] = date_start
                df_city["date_end"] = date_end

                all_dfs.append(df_city)

                print(f"OK - {city} | {year} Q{quarter} | {date_start} -> {date_end} | {len(df_city)} rows")

            except Exception as e:
                print(f"ERROR - {city} | {year} Q{quarter} | {date_start} -> {date_end} | {e}")

if all_dfs:
    Air_Pollution = pd.concat(all_dfs, ignore_index=True)
else:
    Air_Pollution = pd.DataFrame()

Air_Pollution.head()


OK - Paris | 2024 Q1 | 2024-01-01 -> 2024-03-31 | 0 rows
OK - Lyon | 2024 Q1 | 2024-01-01 -> 2024-03-31 | 0 rows
OK - Marseille | 2024 Q1 | 2024-01-01 -> 2024-03-31 | 87633 rows
OK - Paris | 2024 Q2 | 2024-04-01 -> 2024-06-30 | 0 rows
OK - Lyon | 2024 Q2 | 2024-04-01 -> 2024-06-30 | 0 rows
OK - Marseille | 2024 Q2 | 2024-04-01 -> 2024-06-30 | 87633 rows
OK - Paris | 2024 Q3 | 2024-07-01 -> 2024-09-30 | 0 rows
OK - Lyon | 2024 Q3 | 2024-07-01 -> 2024-09-30 | 0 rows
OK - Marseille | 2024 Q3 | 2024-07-01 -> 2024-09-30 | 88596 rows
OK - Paris | 2024 Q4 | 2024-10-01 -> 2024-12-31 | 0 rows
OK - Lyon | 2024 Q4 | 2024-10-01 -> 2024-12-31 | 205836 rows
OK - Marseille | 2024 Q4 | 2024-10-01 -> 2024-12-31 | 88596 rows
OK - Paris | 2025 Q1 | 2025-01-01 -> 2025-03-31 | 0 rows
OK - Lyon | 2025 Q1 | 2025-01-01 -> 2025-03-31 | 363112 rows
OK - Marseille | 2025 Q1 | 2025-01-01 -> 2025-03-31 | 85707 rows
OK - Paris | 2025 Q2 | 2025-04-01 -> 2025-06-30 | 0 rows
OK - Lyon | 2025 Q2 | 2025-04-01 -> 2025-06

,city_target,aasqa_target,year,quarter,date_start,date_end,aasqa,date_maj,code_no2,code_o3,...,epsg_reg,lib_qual,lib_zone,source,type_zone,x_reg,x_wgs84,y_reg,y_wgs84,gml_id2
0,Marseille,93,2024,1,2024-01-01,2024-03-31,93,2025-12-09T18:00:03.412+01:00,1.0,2.0,...,2154,Moyen,Aiglun,AtmoSud,commune,951391.314160,6.139398,6.334355e+06,44.063290,None
1,Marseille,93,2024,1,2024-01-01,2024-03-31,93,2025-12-09T18:00:03.412+01:00,1.0,2.0,...,2154,Moyen,Allemagne-en-Provence,AtmoSud,commune,944147.258483,6.033715,6.303930e+06,43.792143,None
2,Marseille,93,2024,1,2024-01-01,2024-03-31,93,2025-12-09T18:00:03.412+01:00,1.0,2.0,...,2154,Moyen,Allons,AtmoSud,commune,988016.597943,6.590549,6.326299e+06,43.976910,None
3,Marseille,93,2024,1,2024-01-01,2024-03-31,93,2025-12-09T18:00:03.412+01:00,1.0,2.0,...,2154,Moyen,Allos,AtmoSud,commune,989266.331083,6.624810,6.358703e+06,44.268708,None
4,Marseille,93,2024,1,2024-01-01,2024-03-31,93,2025-12-09T18:00:03.412+01:00,1.0,2.0,...,2154,Moyen,Angles,AtmoSud,commune,985182.065619,6.553931,6.322499e+06,43.944256,None


In [174]:
# C - Clean-up Air_Pollution dataset

In [175]:
Air_Pollution.info()

<class 'pandas.DataFrame'>
RangeIndex: 2359517 entries, 0 to 2359516
Data columns (total 28 columns):
 #   Column        Dtype  
---  ------        -----  
 0   city_target   str    
 1   aasqa_target  str    
 2   year          int64  
 3   quarter       int64  
 4   date_start    str    
 5   date_end      str    
 6   aasqa         str    
 7   date_maj      str    
 8   code_no2      float64
 9   code_o3       float64
 10  code_pm10     float64
 11  code_pm25     float64
 12  code_qual     float64
 13  code_so2      float64
 14  code_zone     str    
 15  coul_qual     str    
 16  date_dif      str    
 17  date_ech      str    
 18  epsg_reg      str    
 19  lib_qual      str    
 20  lib_zone      str    
 21  source        str    
 22  type_zone     str    
 23  x_reg         float64
 24  x_wgs84       float64
 25  y_reg         float64
 26  y_wgs84       float64
 27  gml_id2       object 
dtypes: float64(10), int64(2), object(1), str(15)
memory usage: 504.0+ MB


In [176]:
# INITIAL OBSERVATIONS:
# Data availability analysis shows that Paris (including its surrounding areas) cannot be reliably used over this period, due to missing or sparse data (mainly available in Q4 2025).
# Therefore, the analysis will focus on Lyon and Marseille, and will be restricted to 2025 to ensure sufficient data coverage and robustness.
# Additionnaly, this is a large file, to be taken into account for future processing.

In [177]:
# C.1 - Air_Pollution - Keep only Marseille and Lyon cities and surroundings - Remove Paris and surroundings

In [178]:
Air_Pollution = Air_Pollution[
    Air_Pollution["city_target"].isin(["Marseille", "Lyon"])
]

In [179]:
# C.2 - Restrict data to year 2025

In [180]:
Air_Pollution = Air_Pollution[
    Air_Pollution["year"] == 2025
]
Air_Pollution.head()

,city_target,aasqa_target,year,quarter,date_start,date_end,aasqa,date_maj,code_no2,code_o3,...,epsg_reg,lib_qual,lib_zone,source,type_zone,x_reg,x_wgs84,y_reg,y_wgs84,gml_id2
558294,Lyon,84,2025,1,2025-01-01,2025-03-31,84,2025-01-02T13:30:07.447+01:00,1.0,2.0,...,2154,Moyen,L'Abergement-Clémenciat,Atmo Auvergne-Rhône-Alpes,commune,848241.1,4.920886,6563021.4,46.150781,None
558295,Lyon,84,2025,1,2025-01-01,2025-03-31,84,2025-01-02T13:30:07.447+01:00,1.0,2.0,...,2154,Moyen,L'Abergement-de-Varey,Atmo Auvergne-Rhône-Alpes,commune,887495.5,5.423341,6548152.4,46.007215,None
558296,Lyon,84,2025,1,2025-01-01,2025-03-31,84,2025-01-02T13:30:07.447+01:00,1.0,2.0,...,2154,Moyen,Ambérieu-en-Bugey,Atmo Auvergne-Rhône-Alpes,commune,882724.8,5.359568,6542583.8,45.958394,None
558297,Lyon,84,2025,1,2025-01-01,2025-03-31,84,2025-01-02T13:30:07.447+01:00,1.0,2.0,...,2154,Moyen,Ambérieux-en-Dombes,Atmo Auvergne-Rhône-Alpes,commune,847277.1,4.903020,6545792.1,45.995889,None
558298,Lyon,84,2025,1,2025-01-01,2025-03-31,84,2025-01-02T13:30:07.447+01:00,1.0,2.0,...,2154,Moyen,Ambléon,Atmo Auvergne-Rhône-Alpes,commune,902191.7,5.601083,6519791.5,45.747747,None


In [181]:
Air_Pollution.shape

(1799935, 28)

In [182]:
# C.3 - Air_Pollution - Keep only needed columns

In [183]:
cols_to_drop = [
    "aasqa_target",
    "aasqa",
    "epsg_reg",
    "gml_id2",
    "date_start", #remove info added for extraction
    "date_end", #remove info added for extraction
    "date_maj", #remove last update date
    "date_dif", #remove diffusion date, we keep only measurement date
    "coul_qual", #Remove color notion as it can be built later on in a graph if needed
    "code_qual" #Remove code_qual as calculated value - we keep only raw data
]
#color_map = {
#    "Bon": "#50F0E6",
#    "Moyen": "#50CCAA",
#    "Dégradé": "#F0E641",
#    "Mauvais": "#FF5050"
#}
Air_Pollution.drop(
    columns=cols_to_drop,
    inplace=True
)
Air_Pollution.head()

,city_target,year,quarter,code_no2,code_o3,code_pm10,code_pm25,code_so2,code_zone,date_ech,lib_qual,lib_zone,source,type_zone,x_reg,x_wgs84,y_reg,y_wgs84
558294,Lyon,2025,1,1.0,2.0,2.0,2.0,1.0,01001,2025-01-01,Moyen,L'Abergement-Clémenciat,Atmo Auvergne-Rhône-Alpes,commune,848241.1,4.920886,6563021.4,46.150781
558295,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01002,2025-01-01,Moyen,L'Abergement-de-Varey,Atmo Auvergne-Rhône-Alpes,commune,887495.5,5.423341,6548152.4,46.007215
558296,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01004,2025-01-01,Moyen,Ambérieu-en-Bugey,Atmo Auvergne-Rhône-Alpes,commune,882724.8,5.359568,6542583.8,45.958394
558297,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01005,2025-01-01,Moyen,Ambérieux-en-Dombes,Atmo Auvergne-Rhône-Alpes,commune,847277.1,4.903020,6545792.1,45.995889
558298,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01006,2025-01-01,Moyen,Ambléon,Atmo Auvergne-Rhône-Alpes,commune,902191.7,5.601083,6519791.5,45.747747


In [184]:
# C.4 - Air_Pollution - Rename columns

In [185]:
rename_dict = {
    # Measurement zone description
    "city_target": "agglomeration_zone",
    "code_zone": "insee_commune_id",
    "lib_zone": "temp_city_name",
    "type_zone": "zone_type",

    # Date
    "date_ech": "measure_date",

    # pollutants
    "code_no2": "no2_level",          #nitrogen_dioxide
    "code_o3": "o3_level",            #ozone
    "code_pm10": "pm10_level",        
    "code_pm25": "pm25_level",
    "code_so2": "so2_level",          #sulfur_dioxide

    # Global quality
    "lib_qual": "air_quality_label",

    # Geography
    "x_reg": "x_lambert93",
    "y_reg": "y_lambert93",
    "x_wgs84": "longitude",
    "y_wgs84": "latitude",

    # Source
    "source": "data_source"
}
Air_Pollution.rename(
    columns=rename_dict,
    inplace=True
)
Air_Pollution.head()

,agglomeration_zone,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,insee_commune_id,measure_date,air_quality_label,temp_city_name,data_source,zone_type,x_lambert93,longitude,y_lambert93,latitude
558294,Lyon,2025,1,1.0,2.0,2.0,2.0,1.0,01001,2025-01-01,Moyen,L'Abergement-Clémenciat,Atmo Auvergne-Rhône-Alpes,commune,848241.1,4.920886,6563021.4,46.150781
558295,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01002,2025-01-01,Moyen,L'Abergement-de-Varey,Atmo Auvergne-Rhône-Alpes,commune,887495.5,5.423341,6548152.4,46.007215
558296,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01004,2025-01-01,Moyen,Ambérieu-en-Bugey,Atmo Auvergne-Rhône-Alpes,commune,882724.8,5.359568,6542583.8,45.958394
558297,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01005,2025-01-01,Moyen,Ambérieux-en-Dombes,Atmo Auvergne-Rhône-Alpes,commune,847277.1,4.903020,6545792.1,45.995889
558298,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01006,2025-01-01,Moyen,Ambléon,Atmo Auvergne-Rhône-Alpes,commune,902191.7,5.601083,6519791.5,45.747747


In [186]:
#---------------------------------------------------------------------------------------------------
# C.5 - Standardize city labels while preserving surrounding municipalities
#---------------------------------------------------------------------------------------------------

In [186]:
#---------------------------------------------------------------------------------------------------
# - commune_name : analytical grouping label (Lyon, Marseille, or original municipality name)
# - arrondissement: filled only for arrondissement-level records
#---------------------------------------------------------------------------------------------------

In [187]:
# convert in string if needed
codes = Air_Pollution["insee_commune_id"].astype(str)

# column commune_name
Air_Pollution["commune_name"] = Air_Pollution["temp_city_name"]

Air_Pollution.loc[codes.str.startswith("693"), "commune_name"] = "Lyon"
Air_Pollution.loc[codes.str.startswith("132"), "commune_name"] = "Marseille"

# column arrondissement
Air_Pollution["arrondissement"] = None

Air_Pollution.loc[codes.str.startswith("693"), "arrondissement"] = Air_Pollution["temp_city_name"]
Air_Pollution.loc[codes.str.startswith("132"), "arrondissement"] = Air_Pollution["temp_city_name"]

In [188]:
Air_Pollution.head(30)

,agglomeration_zone,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,insee_commune_id,measure_date,air_quality_label,temp_city_name,data_source,zone_type,x_lambert93,longitude,y_lambert93,latitude,commune_name,arrondissement
558294,Lyon,2025,1,1.0,2.0,2.0,2.0,1.0,01001,2025-01-01,Moyen,L'Abergement-Clémenciat,Atmo Auvergne-Rhône-Alpes,commune,848241.1,4.920886,6563021.4,46.150781,L'Abergement-Clémenciat,None
558295,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01002,2025-01-01,Moyen,L'Abergement-de-Varey,Atmo Auvergne-Rhône-Alpes,commune,887495.5,5.423341,6548152.4,46.007215,L'Abergement-de-Varey,None
558296,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01004,2025-01-01,Moyen,Ambérieu-en-Bugey,Atmo Auvergne-Rhône-Alpes,commune,882724.8,5.359568,6542583.8,45.958394,Ambérieu-en-Bugey,None
558297,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01005,2025-01-01,Moyen,Ambérieux-en-Dombes,Atmo Auvergne-Rhône-Alpes,commune,847277.1,4.903020,6545792.1,45.995889,Ambérieux-en-Dombes,None
558298,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01006,2025-01-01,Moyen,Ambléon,Atmo Auvergne-Rhône-Alpes,commune,902191.7,5.601083,6519791.5,45.747747,Ambléon,None
558299,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01008,2025-01-01,Moyen,Ambutrix,Atmo Auvergne-Rhône-Alpes,commune,881030.7,5.336852,6540353.2,45.938770,Ambutrix,None
558300,Lyon,2025,1,1.0,2.0,2.0,2.0,1.0,01009,2025-01-01,Moyen,Condon,Atmo Auvergne-Rhône-Alpes,commune,906247.3,5.655503,6525076.9,45.794100,Condon,None
558301,Lyon,2025,1,1.0,2.0,1.0,1.0,1.0,01011,2025-01-01,Moyen,Petit Vallon,Atmo Auvergne-Rhône-Alpes,commune,904944.6,5.658630,6570969.7,46.207520,Petit Vallon,None
558302,Lyon,2025,1,1.0,2.0,1.0,1.0,1.0,01012,2025-01-01,Moyen,Aranc,Atmo Auvergne-Rhône-Alpes,commune,894055.4,5.507963,6547891.4,46.003022,Aranc,None
558303,Lyon,2025,1,1.0,2.0,1.0,1.0,1.0,01013,2025-01-01,Moyen,Arandas,Atmo Auvergne-Rhône-Alpes,commune,892715.9,5.485838,6536028.4,45.896631,Arandas,None


In [189]:
# QUALITY CHECK

In [190]:
Air_Pollution["arrondissement"].value_counts(dropna=False)

arrondissement
None                            1790919
NaN                                3240
Marseille 1er Arrondissement        361
Marseille 2e Arrondissement         361
Marseille 3e Arrondissement         361
Marseille 4e Arrondissement         361
Marseille 5e Arrondissement         361
Marseille 6e Arrondissement         361
Marseille 7e Arrondissement         361
Marseille 8e Arrondissement         361
Marseille 9e Arrondissement         361
Marseille 10e Arrondissement        361
Marseille 11e Arrondissement        361
Marseille 12e Arrondissement        361
Marseille 13e Arrondissement        361
Marseille 14e Arrondissement        361
Marseille 15e Arrondissement        361
Marseille 16e Arrondissement        361
Name: count, dtype: int64

In [191]:
Air_Pollution[
    Air_Pollution["temp_city_name"].str.contains("Lyon", na=False)
]["temp_city_name"].value_counts()

temp_city_name
Cognat-Lyonne          360
Chazelles-sur-Lyon     360
Lyon                   360
Sainte-Foy-lès-Lyon    360
Name: count, dtype: int64

In [192]:
Air_Pollution[
    Air_Pollution["temp_city_name"].str.contains("Marseille", na=False)
]["temp_city_name"].value_counts()

temp_city_name
Marseille                       361
Marseille 1er Arrondissement    361
Marseille 2e Arrondissement     361
Marseille 3e Arrondissement     361
Marseille 4e Arrondissement     361
Marseille 5e Arrondissement     361
Marseille 6e Arrondissement     361
Marseille 7e Arrondissement     361
Marseille 8e Arrondissement     361
Marseille 9e Arrondissement     361
Marseille 10e Arrondissement    361
Marseille 11e Arrondissement    361
Marseille 12e Arrondissement    361
Marseille 13e Arrondissement    361
Marseille 14e Arrondissement    361
Marseille 15e Arrondissement    361
Marseille 16e Arrondissement    361
Name: count, dtype: int64

In [193]:
# OBSERVATION: The dataset contains mixed spatial granularities: city-level data and arrondissement-level data (for Marseille only). 

In [194]:
# DECISION: To ensure analytical consistency and avoid bias (over-representation of Marseille due to its subdivision into 16 arrondissements), 
# I chose to standardize the dataset at the municipality level.
# ---
# Arrondissement-level data was extracted and stored in a separate dataset for potential fine-grained spatial analysis.

In [195]:
mask_arr = Air_Pollution["temp_city_name"].str.contains("Arrondissement", na=False)

# Backup of arrondissements 
Air_Pollution_Marseille_arrondissements = Air_Pollution[mask_arr].copy()

# Main Dataset stays homogeneously at commune level
Air_Pollution = Air_Pollution[~mask_arr].copy()

In [196]:
Air_Pollution.shape

(1794159, 20)

In [197]:
Air_Pollution_Marseille_arrondissements.head(30)

,agglomeration_zone,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,insee_commune_id,measure_date,air_quality_label,temp_city_name,data_source,zone_type,x_lambert93,longitude,y_lambert93,latitude,commune_name,arrondissement
922049,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13201,2025-01-01,Mauvais,Marseille 1er Arrondissement,AtmoSud,commune,893334.247306,5.381597,6247469.30,43.300147,Marseille,Marseille 1er Arrondissement
922050,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13202,2025-01-01,Mauvais,Marseille 2e Arrondissement,AtmoSud,commune,891361.262651,5.357834,6249688.95,43.321088,Marseille,Marseille 2e Arrondissement
922051,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13203,2025-01-01,Mauvais,Marseille 3e Arrondissement,AtmoSud,commune,893235.564687,5.380948,6248687.15,43.311459,Marseille,Marseille 3e Arrondissement
922052,Marseille,2025,1,2.0,2.0,2.0,3.0,1.0,13204,2025-01-01,Dégradé,Marseille 4e Arrondissement,AtmoSud,commune,894973.725097,5.402213,6248217.40,43.306232,Marseille,Marseille 4e Arrondissement
922053,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13205,2025-01-01,Mauvais,Marseille 5e Arrondissement,AtmoSud,commune,894973.813274,5.401394,6246633.15,43.292617,Marseille,Marseille 5e Arrondissement
922054,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13206,2025-01-01,Mauvais,Marseille 6e Arrondissement,AtmoSud,commune,893260.355967,5.380065,6246071.05,43.287642,Marseille,Marseille 6e Arrondissement
922055,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13207,2025-01-01,Mauvais,Marseille 7e Arrondissement,AtmoSud,commune,891566.292298,5.359098,6245351.10,43.281489,Marseille,Marseille 7e Arrondissement
922056,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13208,2025-01-01,Dégradé,Marseille 8e Arrondissement,AtmoSud,commune,893747.194401,5.384141,6241356.40,43.245277,Marseille,Marseille 8e Arrondissement
922057,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13209,2025-01-01,Dégradé,Marseille 9e Arrondissement,AtmoSud,commune,899737.964412,5.458331,6240607.00,43.236622,Marseille,Marseille 9e Arrondissement
922058,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13210,2025-01-01,Dégradé,Marseille 10e Arrondissement,AtmoSud,commune,897521.047478,5.431563,6244718.00,43.274326,Marseille,Marseille 10e Arrondissement


In [198]:
Air_Pollution_Marseille_arrondissements.shape

(5776, 20)

In [199]:
# C.6 - set back indexes starting from zero

In [200]:
Air_Pollution.reset_index(drop=True, inplace=True)
Air_Pollution.head()

,agglomeration_zone,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,insee_commune_id,measure_date,air_quality_label,temp_city_name,data_source,zone_type,x_lambert93,longitude,y_lambert93,latitude,commune_name,arrondissement
0,Lyon,2025,1,1.0,2.0,2.0,2.0,1.0,01001,2025-01-01,Moyen,L'Abergement-Clémenciat,Atmo Auvergne-Rhône-Alpes,commune,848241.1,4.920886,6563021.4,46.150781,L'Abergement-Clémenciat,None
1,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01002,2025-01-01,Moyen,L'Abergement-de-Varey,Atmo Auvergne-Rhône-Alpes,commune,887495.5,5.423341,6548152.4,46.007215,L'Abergement-de-Varey,None
2,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01004,2025-01-01,Moyen,Ambérieu-en-Bugey,Atmo Auvergne-Rhône-Alpes,commune,882724.8,5.359568,6542583.8,45.958394,Ambérieu-en-Bugey,None
3,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01005,2025-01-01,Moyen,Ambérieux-en-Dombes,Atmo Auvergne-Rhône-Alpes,commune,847277.1,4.903020,6545792.1,45.995889,Ambérieux-en-Dombes,None
4,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01006,2025-01-01,Moyen,Ambléon,Atmo Auvergne-Rhône-Alpes,commune,902191.7,5.601083,6519791.5,45.747747,Ambléon,None


In [201]:
Air_Pollution_Marseille_arrondissements.reset_index(drop=True, inplace=True)
Air_Pollution_Marseille_arrondissements.head()

,agglomeration_zone,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,insee_commune_id,measure_date,air_quality_label,temp_city_name,data_source,zone_type,x_lambert93,longitude,y_lambert93,latitude,commune_name,arrondissement
0,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13201,2025-01-01,Mauvais,Marseille 1er Arrondissement,AtmoSud,commune,893334.247306,5.381597,6247469.30,43.300147,Marseille,Marseille 1er Arrondissement
1,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13202,2025-01-01,Mauvais,Marseille 2e Arrondissement,AtmoSud,commune,891361.262651,5.357834,6249688.95,43.321088,Marseille,Marseille 2e Arrondissement
2,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13203,2025-01-01,Mauvais,Marseille 3e Arrondissement,AtmoSud,commune,893235.564687,5.380948,6248687.15,43.311459,Marseille,Marseille 3e Arrondissement
3,Marseille,2025,1,2.0,2.0,2.0,3.0,1.0,13204,2025-01-01,Dégradé,Marseille 4e Arrondissement,AtmoSud,commune,894973.725097,5.402213,6248217.40,43.306232,Marseille,Marseille 4e Arrondissement
4,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13205,2025-01-01,Mauvais,Marseille 5e Arrondissement,AtmoSud,commune,894973.813274,5.401394,6246633.15,43.292617,Marseille,Marseille 5e Arrondissement


In [202]:
# C.7 - Now, arrondissement column is useless in main DataFrame, and temp_city_name is useless in both DataFrames

In [203]:
Air_Pollution.drop(
    columns=["arrondissement","temp_city_name"],
    inplace=True
)
Air_Pollution.head()

,agglomeration_zone,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,insee_commune_id,measure_date,air_quality_label,data_source,zone_type,x_lambert93,longitude,y_lambert93,latitude,commune_name
0,Lyon,2025,1,1.0,2.0,2.0,2.0,1.0,01001,2025-01-01,Moyen,Atmo Auvergne-Rhône-Alpes,commune,848241.1,4.920886,6563021.4,46.150781,L'Abergement-Clémenciat
1,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01002,2025-01-01,Moyen,Atmo Auvergne-Rhône-Alpes,commune,887495.5,5.423341,6548152.4,46.007215,L'Abergement-de-Varey
2,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01004,2025-01-01,Moyen,Atmo Auvergne-Rhône-Alpes,commune,882724.8,5.359568,6542583.8,45.958394,Ambérieu-en-Bugey
3,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01005,2025-01-01,Moyen,Atmo Auvergne-Rhône-Alpes,commune,847277.1,4.903020,6545792.1,45.995889,Ambérieux-en-Dombes
4,Lyon,2025,1,1.0,2.0,1.0,2.0,1.0,01006,2025-01-01,Moyen,Atmo Auvergne-Rhône-Alpes,commune,902191.7,5.601083,6519791.5,45.747747,Ambléon


In [204]:
Air_Pollution_Marseille_arrondissements.drop(
    columns="temp_city_name",
    inplace=True
)
Air_Pollution_Marseille_arrondissements.head()

,agglomeration_zone,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,insee_commune_id,measure_date,air_quality_label,data_source,zone_type,x_lambert93,longitude,y_lambert93,latitude,commune_name,arrondissement
0,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13201,2025-01-01,Mauvais,AtmoSud,commune,893334.247306,5.381597,6247469.30,43.300147,Marseille,Marseille 1er Arrondissement
1,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13202,2025-01-01,Mauvais,AtmoSud,commune,891361.262651,5.357834,6249688.95,43.321088,Marseille,Marseille 2e Arrondissement
2,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13203,2025-01-01,Mauvais,AtmoSud,commune,893235.564687,5.380948,6248687.15,43.311459,Marseille,Marseille 3e Arrondissement
3,Marseille,2025,1,2.0,2.0,2.0,3.0,1.0,13204,2025-01-01,Dégradé,AtmoSud,commune,894973.725097,5.402213,6248217.40,43.306232,Marseille,Marseille 4e Arrondissement
4,Marseille,2025,1,2.0,2.0,2.0,4.0,1.0,13205,2025-01-01,Mauvais,AtmoSud,commune,894973.813274,5.401394,6246633.15,43.292617,Marseille,Marseille 5e Arrondissement


In [205]:
# C.8 - Re-order columns
# Move insee_commune_id as first column (index) and commune_name in second column

In [206]:
cols = Air_Pollution.columns.tolist()

# remove comumns that will be re-positionned
cols.remove("insee_commune_id")
cols.remove("commune_name")
cols.remove("zone_type")
cols.remove("agglomeration_zone")
cols.remove("measure_date")

# rebuild order
new_cols = ["insee_commune_id", "commune_name","zone_type","agglomeration_zone","measure_date"] + cols

# apply new order of colums
Air_Pollution = Air_Pollution[new_cols]

Air_Pollution.head()

,insee_commune_id,commune_name,zone_type,agglomeration_zone,measure_date,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,air_quality_label,data_source,x_lambert93,longitude,y_lambert93,latitude
0,01001,L'Abergement-Clémenciat,commune,Lyon,2025-01-01,2025,1,1.0,2.0,2.0,2.0,1.0,Moyen,Atmo Auvergne-Rhône-Alpes,848241.1,4.920886,6563021.4,46.150781
1,01002,L'Abergement-de-Varey,commune,Lyon,2025-01-01,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,Atmo Auvergne-Rhône-Alpes,887495.5,5.423341,6548152.4,46.007215
2,01004,Ambérieu-en-Bugey,commune,Lyon,2025-01-01,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,Atmo Auvergne-Rhône-Alpes,882724.8,5.359568,6542583.8,45.958394
3,01005,Ambérieux-en-Dombes,commune,Lyon,2025-01-01,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,Atmo Auvergne-Rhône-Alpes,847277.1,4.903020,6545792.1,45.995889
4,01006,Ambléon,commune,Lyon,2025-01-01,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,Atmo Auvergne-Rhône-Alpes,902191.7,5.601083,6519791.5,45.747747


In [207]:
cols = Air_Pollution_Marseille_arrondissements.columns.tolist()

# retirer les colonnes à repositionner
cols.remove("insee_commune_id")
cols.remove("commune_name")
cols.remove("arrondissement")
cols.remove("zone_type")
cols.remove("agglomeration_zone")
cols.remove("measure_date")

# reconstruire l’ordre
new_cols = ["insee_commune_id", "commune_name","arrondissement","zone_type","agglomeration_zone","measure_date"] + cols

# appliquer
Air_Pollution_Marseille_arrondissements = Air_Pollution_Marseille_arrondissements[new_cols]

Air_Pollution_Marseille_arrondissements.head()

,insee_commune_id,commune_name,arrondissement,zone_type,agglomeration_zone,measure_date,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,air_quality_label,data_source,x_lambert93,longitude,y_lambert93,latitude
0,13201,Marseille,Marseille 1er Arrondissement,commune,Marseille,2025-01-01,2025,1,2.0,2.0,2.0,4.0,1.0,Mauvais,AtmoSud,893334.247306,5.381597,6247469.30,43.300147
1,13202,Marseille,Marseille 2e Arrondissement,commune,Marseille,2025-01-01,2025,1,2.0,2.0,2.0,4.0,1.0,Mauvais,AtmoSud,891361.262651,5.357834,6249688.95,43.321088
2,13203,Marseille,Marseille 3e Arrondissement,commune,Marseille,2025-01-01,2025,1,2.0,2.0,2.0,4.0,1.0,Mauvais,AtmoSud,893235.564687,5.380948,6248687.15,43.311459
3,13204,Marseille,Marseille 4e Arrondissement,commune,Marseille,2025-01-01,2025,1,2.0,2.0,2.0,3.0,1.0,Dégradé,AtmoSud,894973.725097,5.402213,6248217.40,43.306232
4,13205,Marseille,Marseille 5e Arrondissement,commune,Marseille,2025-01-01,2025,1,2.0,2.0,2.0,4.0,1.0,Mauvais,AtmoSud,894973.813274,5.401394,6246633.15,43.292617


In [208]:
# C.9 - Remove additional useless columns

In [209]:
Air_Pollution["zone_type"].value_counts(dropna=False)

zone_type
commune    1794159
Name: count, dtype: int64

In [210]:
Air_Pollution_Marseille_arrondissements["zone_type"].value_counts(dropna=False)

zone_type
commune    5776
Name: count, dtype: int64

In [211]:
# Remove useless column zone_type

In [212]:
Air_Pollution.drop(
    columns="zone_type",
    inplace=True
)
Air_Pollution.head()

,insee_commune_id,commune_name,agglomeration_zone,measure_date,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,air_quality_label,data_source,x_lambert93,longitude,y_lambert93,latitude
0,01001,L'Abergement-Clémenciat,Lyon,2025-01-01,2025,1,1.0,2.0,2.0,2.0,1.0,Moyen,Atmo Auvergne-Rhône-Alpes,848241.1,4.920886,6563021.4,46.150781
1,01002,L'Abergement-de-Varey,Lyon,2025-01-01,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,Atmo Auvergne-Rhône-Alpes,887495.5,5.423341,6548152.4,46.007215
2,01004,Ambérieu-en-Bugey,Lyon,2025-01-01,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,Atmo Auvergne-Rhône-Alpes,882724.8,5.359568,6542583.8,45.958394
3,01005,Ambérieux-en-Dombes,Lyon,2025-01-01,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,Atmo Auvergne-Rhône-Alpes,847277.1,4.903020,6545792.1,45.995889
4,01006,Ambléon,Lyon,2025-01-01,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,Atmo Auvergne-Rhône-Alpes,902191.7,5.601083,6519791.5,45.747747


In [213]:
Air_Pollution_Marseille_arrondissements.drop(
    columns="zone_type",
    inplace=True
)
Air_Pollution_Marseille_arrondissements.head()

,insee_commune_id,commune_name,arrondissement,agglomeration_zone,measure_date,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,air_quality_label,data_source,x_lambert93,longitude,y_lambert93,latitude
0,13201,Marseille,Marseille 1er Arrondissement,Marseille,2025-01-01,2025,1,2.0,2.0,2.0,4.0,1.0,Mauvais,AtmoSud,893334.247306,5.381597,6247469.30,43.300147
1,13202,Marseille,Marseille 2e Arrondissement,Marseille,2025-01-01,2025,1,2.0,2.0,2.0,4.0,1.0,Mauvais,AtmoSud,891361.262651,5.357834,6249688.95,43.321088
2,13203,Marseille,Marseille 3e Arrondissement,Marseille,2025-01-01,2025,1,2.0,2.0,2.0,4.0,1.0,Mauvais,AtmoSud,893235.564687,5.380948,6248687.15,43.311459
3,13204,Marseille,Marseille 4e Arrondissement,Marseille,2025-01-01,2025,1,2.0,2.0,2.0,3.0,1.0,Dégradé,AtmoSud,894973.725097,5.402213,6248217.40,43.306232
4,13205,Marseille,Marseille 5e Arrondissement,Marseille,2025-01-01,2025,1,2.0,2.0,2.0,4.0,1.0,Mauvais,AtmoSud,894973.813274,5.401394,6246633.15,43.292617


In [214]:
# DATASET STRUCTURE NOTE
# The main dataset is kept at municipality level.
# "commune_name" refers to the actual municipality name.
# "agglomeration_zone" refers to the target metropolitan area used in the source data
# (e.g. Marseille or Lyon), and may include surrounding municipalities.
#
# Marseille arrondissement-level data was isolated in a separate dataframe
# to preserve a homogeneous granularity in the main dataset while keeping
# the possibility of finer spatial analysis for Marseille.

In [215]:
# C.10 - QUALITY CHECKS for NULL values

In [216]:
Air_Pollution.info()

<class 'pandas.DataFrame'>
RangeIndex: 1794159 entries, 0 to 1794158
Data columns (total 17 columns):
 #   Column              Dtype  
---  ------              -----  
 0   insee_commune_id    str    
 1   commune_name        str    
 2   agglomeration_zone  str    
 3   measure_date        str    
 4   year                int64  
 5   quarter             int64  
 6   no2_level           float64
 7   o3_level            float64
 8   pm10_level          float64
 9   pm25_level          float64
 10  so2_level           float64
 11  air_quality_label   str    
 12  data_source         str    
 13  x_lambert93         float64
 14  longitude           float64
 15  y_lambert93         float64
 16  latitude            float64
dtypes: float64(9), int64(2), str(6)
memory usage: 232.7 MB


In [217]:
#Select empty lines
Air_Pollution.isna().sum()

insee_commune_id         0
commune_name          1709
agglomeration_zone       0
measure_date             0
year                     0
quarter                  0
no2_level                0
o3_level                 0
pm10_level               0
pm25_level            5001
so2_level                0
air_quality_label        0
data_source              0
x_lambert93           4949
longitude             4949
y_lambert93           4949
latitude              4949
dtype: int64

In [218]:
Air_Pollution[
    Air_Pollution["commune_name"].isna()
].head()

,insee_commune_id,commune_name,agglomeration_zone,measure_date,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,air_quality_label,data_source,x_lambert93,longitude,y_lambert93,latitude
20180,15031,NaN,Lyon,2025-03-04,2025,1,1.0,1.0,1.0,1.0,1.0,Bon,Atmo Auvergne-Rhône-Alpes,NaN,NaN,NaN,NaN
20181,15035,NaN,Lyon,2025-03-04,2025,1,1.0,1.0,1.0,1.0,1.0,Bon,Atmo Auvergne-Rhône-Alpes,NaN,NaN,NaN,NaN
20182,15047,NaN,Lyon,2025-03-04,2025,1,1.0,1.0,1.0,1.0,1.0,Bon,Atmo Auvergne-Rhône-Alpes,NaN,NaN,NaN,NaN
20183,15171,NaN,Lyon,2025-03-04,2025,1,1.0,1.0,1.0,1.0,1.0,Bon,Atmo Auvergne-Rhône-Alpes,NaN,NaN,NaN,NaN
20210,15031,NaN,Lyon,2025-03-03,2025,1,1.0,1.0,1.0,1.0,1.0,Bon,Atmo Auvergne-Rhône-Alpes,NaN,NaN,NaN,NaN


In [219]:
# Fill empty commune names thanks to table insee_to_commune (left join)

In [220]:
Air_Pollution = (
    Air_Pollution
    .merge(
        insee_to_commune[["insee_commune_id", "commune_name"]],
        on="insee_commune_id",
        how="left",
        suffixes=("", "_ref")
    )
)

Air_Pollution["commune_name"] = Air_Pollution["commune_name"].fillna(
    Air_Pollution["commune_name_ref"]
)

Air_Pollution.drop(columns="commune_name_ref", inplace=True)

In [221]:
Air_Pollution[
    Air_Pollution["commune_name"].isna()
].head()

,insee_commune_id,commune_name,agglomeration_zone,measure_date,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,air_quality_label,data_source,x_lambert93,longitude,y_lambert93,latitude
363415,05120,NaN,Marseille,2025-01-01,2025,1,1.0,2.0,1.0,1.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
364362,05120,NaN,Marseille,2025-01-02,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
365309,05120,NaN,Marseille,2025-01-03,2025,1,1.0,2.0,1.0,1.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
366256,05120,NaN,Marseille,2025-01-04,2025,1,1.0,2.0,1.0,1.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
367203,05120,NaN,Marseille,2025-01-05,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN


In [222]:
# how many empty communes
Air_Pollution["commune_name"].isna().sum()

np.int64(361)

In [223]:
# how many unique insee ids that couldn't be found into insee_to_commune table
Air_Pollution.loc[
    Air_Pollution["commune_name"].isna(),
    "insee_commune_id"
].nunique()

1

In [224]:
# OBSERVATION: only one INSEE code (05120) could not be found in the insee_to_commune reference table, indicating a potential gap in the reference dataset.
# It corresponds to Saint-Martin-de-Queyrières (Hautes-Alpes) and will be added to the reference table, as well as used to update the Air_Pollution dataset.

In [225]:
missing_commune = pd.DataFrame({
    "insee_commune_id": ["05120"],
    "commune_name": ["Saint-Martin-de-Queyrières"]
})

insee_to_commune = pd.concat(
    [insee_to_commune, missing_commune],
    ignore_index=True
)
insee_to_commune[
    insee_to_commune["insee_commune_id"] == "05120"
]

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage
34875,05120,NaN,Saint-Martin-de-Queyrières,NaN,NaN,NaN,NaN


In [226]:
Air_Pollution.loc[
    Air_Pollution["insee_commune_id"] == "05120",
    "commune_name"
] = "Saint-Martin-de-Queyrières"
Air_Pollution[
    Air_Pollution["insee_commune_id"] == "05120"
]

,insee_commune_id,commune_name,agglomeration_zone,measure_date,year,quarter,no2_level,o3_level,pm10_level,pm25_level,so2_level,air_quality_label,data_source,x_lambert93,longitude,y_lambert93,latitude
363415,05120,Saint-Martin-de-Queyrières,Marseille,2025-01-01,2025,1,1.0,2.0,1.0,1.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
364362,05120,Saint-Martin-de-Queyrières,Marseille,2025-01-02,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
365309,05120,Saint-Martin-de-Queyrières,Marseille,2025-01-03,2025,1,1.0,2.0,1.0,1.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
366256,05120,Saint-Martin-de-Queyrières,Marseille,2025-01-04,2025,1,1.0,2.0,1.0,1.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
367203,05120,Saint-Martin-de-Queyrières,Marseille,2025-01-05,2025,1,1.0,2.0,1.0,2.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1789727,05120,Saint-Martin-de-Queyrières,Marseille,2025-12-22,2025,4,1.0,2.0,1.0,NaN,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
1790674,05120,Saint-Martin-de-Queyrières,Marseille,2025-12-23,2025,4,1.0,2.0,1.0,1.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
1791621,05120,Saint-Martin-de-Queyrières,Marseille,2025-12-24,2025,4,1.0,2.0,1.0,1.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN
1792568,05120,Saint-Martin-de-Queyrières,Marseille,2025-12-28,2025,4,1.0,2.0,1.0,1.0,1.0,Moyen,AtmoSud,NaN,NaN,NaN,NaN


In [227]:
Air_Pollution.isna().sum()

insee_commune_id         0
commune_name             0
agglomeration_zone       0
measure_date             0
year                     0
quarter                  0
no2_level                0
o3_level                 0
pm10_level               0
pm25_level            5001
so2_level                0
air_quality_label        0
data_source              0
x_lambert93           4949
longitude             4949
y_lambert93           4949
latitude              4949
dtype: int64

In [228]:
#reorder insee_to_commune reference table by insee_commune_id and reset index

In [229]:
insee_to_commune = insee_to_commune.sort_values(by="insee_commune_id")
insee_to_commune = (
    insee_to_commune
    .sort_values(by="insee_commune_id")
    .reset_index(drop=True)
)
insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False,False,False,0.0
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False,False,False,0.0
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True,True,True,1.0
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True,False,False,0.0
4,01006,AMBLEON,Ambléon,False,False,False,0.0


In [230]:
Air_Pollution_Marseille_arrondissements.info()

<class 'pandas.DataFrame'>
RangeIndex: 5776 entries, 0 to 5775
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   insee_commune_id    5776 non-null   str    
 1   commune_name        5776 non-null   str    
 2   arrondissement      5776 non-null   object 
 3   agglomeration_zone  5776 non-null   str    
 4   measure_date        5776 non-null   str    
 5   year                5776 non-null   int64  
 6   quarter             5776 non-null   int64  
 7   no2_level           5776 non-null   float64
 8   o3_level            5776 non-null   float64
 9   pm10_level          5776 non-null   float64
 10  pm25_level          5776 non-null   float64
 11  so2_level           5776 non-null   float64
 12  air_quality_label   5776 non-null   str    
 13  data_source         5776 non-null   str    
 14  x_lambert93         5776 non-null   float64
 15  longitude           5776 non-null   float64
 16  y_lambert93      

In [231]:
# All GOOD for this DataFrame : no NULL values, and dtypes OK

In [232]:
# C.11 - DECISION: investigate the availability of data for INSEE code "05120" across the crime and revenue datasets.

In [233]:
declared_income_2021.head()

,iris_id,commune_name,place_in_commune,menages_fiscaux_imposes_pct,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_autres_revenus_pct
0,010040101,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,43.0,29.0,12610.0,19330.0,26390.0,7760.0,33850.0,69.3,27.1,3.6
1,010040102,Ambérieu-en-Bugey,Longeray-Gare,42.0,39.0,9730.0,16830.0,24620.0,4680.0,33030.0,70.5,25.6,3.9
2,010040201,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,47.0,29.0,12220.0,19940.0,27650.0,6500.0,37220.0,69.3,26.7,4.0
3,010040202,Ambérieu-en-Bugey,Tiret-Les Allymes,62.0,14.0,18350.0,25560.0,35010.0,11960.0,45520.0,65.5,21.8,12.7
4,010330102,Valserhône,Centre Ville,42.0,31.0,11280.0,19870.0,30050.0,5250.0,44820.0,74.1,23.0,2.9


In [234]:
code = "05120"

print("Declared income:")
print(declared_income_2021.loc[
    declared_income_2021["iris_id"].astype(str).str[:5] == code
])

print("\nDisposable income:")
print(disposable_income_2021.loc[
    disposable_income_2021["iris_id"].astype(str).str[:5] == code
])

Declared income:
Empty DataFrame
Columns: [iris_id, commune_name, place_in_commune, menages_fiscaux_imposes_pct, taux_bas_revenus_pct, revenu_q1_eur, revenu_median_eur, revenu_q3_eur, revenu_d1_eur, revenu_d9_eur, part_revenus_activite_pct, part_pensions_pct, part_autres_revenus_pct]
Index: []

Disposable income:
Empty DataFrame
Columns: [iris_id, commune_name, place_in_commune, taux_bas_revenus_pct, revenu_q1_eur, revenu_median_eur, revenu_q3_eur, revenu_d1_eur, revenu_d9_eur, part_revenus_activite_pct, part_pensions_pct, part_revenus_patrimoine_pct, part_prestations_sociales_pct, part_prestations_familiales_pct, part_minima_sociaux_pct, part_prestations_logement_pct, part_impots_pct]
Index: []


In [235]:
crime.head()

,insee_commune_id,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref,data_available
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,0.0,767,True
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,767,False
2,01001,2016,Violences sexuelles,Victime,0.0,0.0,767,True
3,01001,2016,Vols avec armes,Infraction,0.0,0.0,767,True
4,01001,2016,Vols violents sans arme,Infraction,0.0,0.0,767,True


In [236]:
code = "05120"

print("Crime:")
print(crime.loc[
    crime["insee_commune_id"].astype(str) == code
])

Crime:
Empty DataFrame
Columns: [insee_commune_id, annee, crime_type, unite_de_compte, nombre, taux_pour_mille, insee_pop_ref, data_available]
Index: []


In [237]:
# OBSERVATION : no revenue or crime data is available for INSEE code "05120".
# The corresponding availability flags in the insee_to_commune table will be updated accordingly.

In [238]:
code = "05120"

mask = insee_to_commune["insee_commune_id"].astype(str) == code

updates = {
    "has_income_data_any": False,
    "has_income_data_all": False,
    "income_data_coverage": 0,
    "has_crime_data": False
}

for col, value in updates.items():
    if col in insee_to_commune.columns:
        insee_to_commune.loc[mask, col] = value

insee_to_commune.loc[mask]

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage
1808,05120,NaN,Saint-Martin-de-Queyrières,False,False,False,0.0


In [239]:
#fill in commune_name_upper
code = "05120"

mask = insee_to_commune["insee_commune_id"].astype(str) == code

insee_to_commune.loc[mask, "commune_name_upper"] = (
    insee_to_commune.loc[mask, "commune_name"].str.upper()
)
insee_to_commune.loc[mask, ["commune_name", "commune_name_upper"]]

,commune_name,commune_name_upper
1808,Saint-Martin-de-Queyrières,SAINT-MARTIN-DE-QUEYRIÈRES


In [240]:
# C.12 - DECISION : add an Air_Pollution availability flag to the INSEE reference table for all communes

In [241]:
# Get all commune IDs present in the Air_Pollution dataset
insee_with_air_pollution = set(Air_Pollution["insee_commune_id"].astype(str)) # set() automatically removes duplicates

# Add Air_Pollution availability flag to the reference table
insee_to_commune["has_air_pollution_data"] = (
    insee_to_commune["insee_commune_id"].astype(str).isin(insee_with_air_pollution)
)

insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage,has_air_pollution_data
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False,False,False,0.0,True
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False,False,False,0.0,True
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True,True,True,1.0,True
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True,False,False,0.0,True
4,01006,AMBLEON,Ambléon,False,False,False,0.0,True


In [242]:
insee_to_commune.shape

(34876, 8)

In [243]:
insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage,has_air_pollution_data
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False,False,False,0.0,True
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False,False,False,0.0,True
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True,True,True,1.0,True
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True,False,False,0.0,True
4,01006,AMBLEON,Ambléon,False,False,False,0.0,True


In [244]:
# C.13 - DATA SUFFICIENCY CHECKS:
# The following data would provide a quick win to enhance data exploration:
# A - POPULATION DENSITY, a key variable to explore potential relationships with crime, income, and air pollution.
# B - ESTABLISHMENTS COUNTS BY MAJOR ACTIVITY SECTORS - These variables can be used as simple proxies for local economic structure
# and may help explore relationships with crime, income, and air pollution.

In [245]:
# C.13.1 - Add POPULATION DENSITY in insee_to_commune reference table

In [246]:
url = "https://www.insee.fr/fr/statistiques/fichier/8571524/fichier_diffusion_2025.xlsx"

# Read INSEE density table (2025)
density = pd.read_excel(
    url,
    sheet_name="Maille communale",
    skiprows=4
)

# Keep only useful columns
density = density[["CODGEO", "DENS_AAV", "LIBDENS_AAV"]].copy()
density = density.rename(columns={
    "CODGEO": "insee_commune_id",
    "DENS_AAV": "density_class",
    "LIBDENS_AAV": "density_label"
})
density.head()

,insee_commune_id,density_class,density_label
0,01001,4,Rural non périurbain
1,01002,4,Rural non périurbain
2,01004,2,Urbain intermédiaire
3,01005,3,Rural périurbain
4,01006,4,Rural non périurbain


In [247]:
# Harmonize join key - Pad with leading zeros to ensure a length of 5 characters
density["insee_commune_id"] = density["insee_commune_id"].astype(str).str.zfill(5)
insee_to_commune["insee_commune_id"] = (
    insee_to_commune["insee_commune_id"].astype(str).str.zfill(5) 
)

# Join to reference table
insee_to_commune = insee_to_commune.merge(
    density,
    on="insee_commune_id",
    how="left"
)

insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage,has_air_pollution_data,density_class,density_label
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False,False,False,0.0,True,4.0,Rural non périurbain
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False,False,False,0.0,True,4.0,Rural non périurbain
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True,True,True,1.0,True,2.0,Urbain intermédiaire
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True,False,False,0.0,True,3.0,Rural périurbain
4,01006,AMBLEON,Ambléon,False,False,False,0.0,True,4.0,Rural non périurbain


In [248]:
# C.13.2 - Add ESTABLISHMENTS COUNTS BY MAJOR ACTIVITY SECTORS in insee_to_commune reference table

In [250]:
# Download and extract FLORES A5 Excel dataset
url = "https://www.insee.fr/fr/statistiques/fichier/8266010/fichiers_activites_effectifs_A5-XLSX.zip"
zip_path = "source_files/Economy/fichiers_activites_effectifs_A5-XLSX.zip"
extract_path = "source_files/Economy/extracted"

download_and_extract_zip(url, zip_path, extract_path)

# Find the extracted Excel file
xlsx_files = [f for f in os.listdir(extract_path) if f.lower().endswith(".xlsx")]

if not xlsx_files:
    raise FileNotFoundError("No Excel file found in extracted FLORES archive.")

target_file = "FLORES_ETABLISSEMENTS_A5_fr.xlsx"

if target_file in xlsx_files:
    economy_path = os.path.join(extract_path, target_file)
else:
    raise FileNotFoundError("FLORES_ETABLISSEMENTS_A5_fr.xlsx NOT found in extracted FLORES archive.")

# Read ENSEMBLE sheet without header
raw_economy = pd.read_excel(
    economy_path,
    sheet_name="ENSEMBLE",
    header=None,
    dtype=str
)

# Detect the row containing sector codes: AZ, BE, FZ, GU, OQ
header_candidates = raw_economy.index[
    raw_economy.apply(
        lambda row: {"AZ", "BE", "FZ", "GU", "OQ"}.issubset(set(row.dropna().astype(str))),
        axis=1
    )
]

if len(header_candidates) == 0:
    raise ValueError(
        "No header row containing AZ, BE, FZ, GU and OQ was found in sheet 'ENSEMBLE'."
    )

header_row_idx = header_candidates[0]

# Rebuild dataframe with this row as header
economy = raw_economy.iloc[header_row_idx:].copy()
economy.columns = economy.iloc[0]
economy = economy.iloc[1:].reset_index(drop=True)

economy.head()

Download and extraction complete


C:\Users\mulau\anaconda3\envs\data_env\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


3,NaN,NaN,AZ,BE,FZ,GU,OQ,_T
0,NaN,NaN,"Agriculture, sylviculture et pêche","Industrie manufacturière, industries extractiv...",Construction,Services principalement marchands,"Administration publique, enseignement, santé h...",Total
1,01001,L'Abergement-Clémenciat,4,0,3,10,2,19
2,01002,L'Abergement-de-Varey,1,0,0,2,1,4
3,01004,Ambérieu-en-Bugey,2,33,57,411,89,592
4,01005,Ambérieux-en-Dombes,2,4,10,25,8,49


In [251]:
# Remove the descriptive row just below the header if still present
economy = economy[
    economy.iloc[:, 0].astype(str).str.strip().str.match(r"^\d{5}$", na=False)
].reset_index(drop=True)

economy.head()

3,NaN,NaN,AZ,BE,FZ,GU,OQ,_T
0,01001,L'Abergement-Clémenciat,4,0,3,10,2,19
1,01002,L'Abergement-de-Varey,1,0,0,2,1,4
2,01004,Ambérieu-en-Bugey,2,33,57,411,89,592
3,01005,Ambérieux-en-Dombes,2,4,10,25,8,49
4,01006,Ambléon,0,0,0,0,1,1


In [252]:
# Complete Column names
economy.columns.values[0] = "insee_commune_id"
economy.columns.values[1] = "commune_name"

economy.columns

Index(['insee_commune_id', 'commune_name', 'AZ', 'BE', 'FZ', 'GU', 'OQ', '_T'], dtype='str', name=3)

In [253]:
economy.info()

<class 'pandas.DataFrame'>
RangeIndex: 34515 entries, 0 to 34514
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   insee_commune_id  34515 non-null  str  
 1   commune_name      34515 non-null  str  
 2   AZ                34511 non-null  str  
 3   BE                34511 non-null  str  
 4   FZ                34511 non-null  str  
 5   GU                34511 non-null  str  
 6   OQ                34511 non-null  str  
 7   _T                34511 non-null  str  
dtypes: str(8)
memory usage: 2.1 MB


In [254]:
# Keep only useful columns
economy = economy[[
    "insee_commune_id",
    "AZ",
    "BE",
    "FZ",
    "GU",
    "OQ",
    "_T"
]].copy()

# Harmonize commune code
# Pad with leading zeros to ensure a length of 5 characters
economy["insee_commune_id"] = economy["insee_commune_id"].astype(str).str.strip().str.zfill(5)
insee_to_commune["insee_commune_id"] = (
    insee_to_commune["insee_commune_id"].astype(str).str.strip().str.zfill(5)
)

# Rename columns for clarity
economy.rename(columns={
    "AZ": "count_agriculture",
    "BE": "count_industry",
    "FZ": "count_construction",
    "GU": "count_commercial_services",
    "OQ": "count_public_services",
    "_T": "total_count"
}, inplace=True)

economy.head()

3,insee_commune_id,count_agriculture,count_industry,count_construction,count_commercial_services,count_public_services,total_count
0,01001,4,0,3,10,2,19
1,01002,1,0,0,2,1,4
2,01004,2,33,57,411,89,592
3,01005,2,4,10,25,8,49
4,01006,0,0,0,0,1,1


In [255]:
# Convert establishment counts to numeric
sector_cols = [
    "count_agriculture",
    "count_industry",
    "count_construction",
    "count_commercial_services",
    "count_public_services",
    "total_count"
]

for col in sector_cols:
    economy[col] = pd.to_numeric(economy[col], errors="coerce")

economy.info()

<class 'pandas.DataFrame'>
RangeIndex: 34515 entries, 0 to 34514
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   insee_commune_id           34515 non-null  str    
 1   count_agriculture          34511 non-null  float64
 2   count_industry             34511 non-null  float64
 3   count_construction         34511 non-null  float64
 4   count_commercial_services  34511 non-null  float64
 5   count_public_services      34511 non-null  float64
 6   total_count                34511 non-null  float64
dtypes: float64(6), str(1)
memory usage: 1.8 MB


In [256]:
# Merge FLORES sector data into the reference table
insee_to_commune = insee_to_commune.merge(
    economy[["insee_commune_id"] + sector_cols],
    on="insee_commune_id",
    how="left"
)

# Quick check
insee_to_commune.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage,has_air_pollution_data,density_class,density_label,count_agriculture,count_industry,count_construction,count_commercial_services,count_public_services,total_count
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False,False,False,0.0,True,4.0,Rural non périurbain,4.0,0.0,3.0,10.0,2.0,19.0
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False,False,False,0.0,True,4.0,Rural non périurbain,1.0,0.0,0.0,2.0,1.0,4.0
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True,True,True,1.0,True,2.0,Urbain intermédiaire,2.0,33.0,57.0,411.0,89.0,592.0
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True,False,False,0.0,True,3.0,Rural périurbain,2.0,4.0,10.0,25.0,8.0,49.0
4,01006,AMBLEON,Ambléon,False,False,False,0.0,True,4.0,Rural non périurbain,0.0,0.0,0.0,0.0,1.0,1.0


In [257]:
# Identify communes with missing values in economic sector indicators
# (agriculture, industry, construction, commercial services, public services, total)
insee_to_commune["economy_data_status"] = "complete"

insee_to_commune.loc[
    insee_to_commune[sector_cols].isna().all(axis=1),
    "economy_data_status"
] = "no_data"

insee_to_commune.loc[
    insee_to_commune[sector_cols].isna().any(axis=1) &
    ~insee_to_commune[sector_cols].isna().all(axis=1),
    "economy_data_status"
] = "partial_data"

print(insee_to_commune["economy_data_status"].value_counts())

insee_to_commune.head()

economy_data_status
complete    34511
no_data       365
Name: count, dtype: int64


,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage,has_air_pollution_data,density_class,density_label,count_agriculture,count_industry,count_construction,count_commercial_services,count_public_services,total_count,economy_data_status
0,01001,ABERGEMENT CLEMENCIAT,Abergement-Clémenciat,False,False,False,0.0,True,4.0,Rural non périurbain,4.0,0.0,3.0,10.0,2.0,19.0,complete
1,01002,ABERGEMENT DE VAREY,Abergement-de-Varey,False,False,False,0.0,True,4.0,Rural non périurbain,1.0,0.0,0.0,2.0,1.0,4.0,complete
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True,True,True,1.0,True,2.0,Urbain intermédiaire,2.0,33.0,57.0,411.0,89.0,592.0,complete
3,01005,AMBERIEUX EN DOMBES,Ambérieux-en-Dombes,True,False,False,0.0,True,3.0,Rural périurbain,2.0,4.0,10.0,25.0,8.0,49.0,complete
4,01006,AMBLEON,Ambléon,False,False,False,0.0,True,4.0,Rural non périurbain,0.0,0.0,0.0,0.0,1.0,1.0,complete


In [258]:
# C.14 - FINAL DATA QUALITY CHECK: assess data coverage at commune level
# Count how many communes have:
# - full coverage across all datasets
# - partial coverage across datasets
# - no available data at all

In [259]:
# Boolean columns used to assess commune-level data coverage
coverage_cols = [
    "has_crime_data",
    "has_income_data_any",
    "has_air_pollution_data"
]

# Count how many main data sources are available for each commune
insee_to_commune["available_source_count"] = (
    insee_to_commune[coverage_cols]
    .sum(axis=1)
)

# Create coverage category
insee_to_commune["data_coverage_status"] = "partial_data"

# Complete coverage:
# - all three main sources available
# - density information available
# - economy data available
insee_to_commune.loc[
    (
        (insee_to_commune["has_crime_data"] == True) &
        (insee_to_commune["has_income_data_any"] == True) &
        (insee_to_commune["has_air_pollution_data"] == True) &
        (insee_to_commune["density_class"] > 0) &
        (insee_to_commune["economy_data_status"] != "no_data")
    ),
    "data_coverage_status"
] = "fully_complete"

# No data:
# - none of the three main sources available
# - no density information
# - no economy data
insee_to_commune.loc[
    (
        (insee_to_commune["has_crime_data"] == False) &
        (insee_to_commune["has_income_data_any"] == False) &
        (insee_to_commune["has_air_pollution_data"] == False) 
    ),
    "data_coverage_status"
] = "no_core_data"

In [260]:
coverage_summary = (
    insee_to_commune["data_coverage_status"]
    .value_counts()
    .rename_axis("coverage_status")
    .reset_index(name="commune_count")
)

coverage_summary

,coverage_status,commune_count
0,no_core_data,20778
1,partial_data,13736
2,fully_complete,362


In [ ]:
# D - Final save of cleaned files

In [261]:
known_crimes.to_csv("cleaned_data/known_crimes_clean.csv", index=False)
crime.to_csv("cleaned_data/crime_clean.csv", index=False)
declared_income_2021.to_csv("cleaned_data/declared_income_2021_clean.csv", index=False)
disposable_income_2021.to_csv("cleaned_data/disposable_income_2021_clean.csv", index=False)

In [262]:
insee_to_commune.to_csv("cleaned_data/insee_to_commune_clean.csv", index=False)
iris_to_commune.to_csv("cleaned_data/iris_to_commune_clean.csv", index=False)

In [263]:
economy.to_csv("cleaned_data/economy_clean.csv", index=False)
Air_Pollution.to_csv("cleaned_data/air_pollution_clean.csv", index=False)
Air_Pollution_Marseille_arrondissements.to_csv("cleaned_data/air_pollution_marseille_arrondissements_clean.csv", index=False)

In [ ]:
# E - FINAL OBSERVATIONS AND ANALYTICAL DECISIONS

In [264]:
# This notebook delivers the data preparation layer required for multi-indicator territorial analysis.
# N3 focuses on collecting, cleaning and harmonizing heterogeneous public datasets related to crime, income, economic activity, air quality and geographic correspondences. 
# The result is a set of structured analytical tables, exported into the `cleaned_data/` folder and ready for downstream aggregation.

# Beyond standard data cleaning, the main challenge addressed in this phase is cross-source consistency. 
# The datasets differ in format, granularity, temporal reference and geographic coverage. 
# Particular attention is given to type normalization, key alignment (INSEE commune id) and the construction of a core correspondence table (`insee_to_commune`) enabling joins across different data sources.

In [ ]:
# LIMITATIONS OF THE STUDY
# ------------------------
# Beyond data preparation challenges, a key limitation lies in cross-source heterogeneity.
# The datasets used in this analysis differ in both geographic coverage and temporal reference, which affects comparability.

# From a geographic perspective:
# - Air pollution data (2025) is limited to the Marseille and Lyon areas, resulting in a reduced subset of 362 communes with complete data across all variables.
# - Other datasets (income, crime, economic activity, density) cover the national territory, although completeness may vary depending on data availability and reporting constraints (notably for crime data).

# From a temporal perspective:
# - Income data refers to 2021 and may not reflect current economic conditions.
# - Crime data spans multiple years (2016–2024), but is partially incomplete due to confidentiality constraints.
# - Air pollution data refers to 2025.
# - Economic activity data (establishment counts) refers to 2024.
# - IRIS geographic codes are based on 2021 and are considered relatively stable.
# - INSEE commune identifiers (2025) are stable and consistent over time.

In [297]:
# METHODOLOGY FOR NEXT STEPS
# -------------------------
# The next analytical phase (N4) will be conducted using a controlled and fully documented subset, prioritizing:
# - data completeness
# - cross-indicator comparability
# - interpretability of results

# This approach strengthens the reliability of subsequent exploratory analysis, scoring logic and multi-criteria assessment.

# NEXT PHASES (STILL ON-GOING)
# -----------
# N4 — Data aggregation at commune level and initial exploration:
# aggregating all retained indicators into a single analytical table at commune level, in preparation for multi-criteria exploration and decision-support analysis.

# N5 — Multi-criteria analysis

# METHODOLOGICAL CHOICES
# ---------------------
# The next analysis phase will be restricted to 362 communes with complete data across all variables, ensuring internal consistency of the dataset.
# However, this subset is geographically biased (Marseille and Lyon areas) and temporally heterogeneous.
# Results should therefore be interpreted with caution and not generalized to the entire national territory.

In [ ]:
# ============================================================
# F - FINAL PREPARATION FOR N4:
# create a mirrored subset restricted to fully complete communes
# ============================================================

In [ ]:
# F.1 - New repository for final material

In [265]:
#create new folder
subset_dir = "cleaned_data_complete_subset"
os.makedirs(subset_dir, exist_ok=True)

In [ ]:
# F.2 - Create the set of complete communes

In [266]:
# Reference list of the 362 fully complete communes
complete_communes = insee_to_commune.loc[
    insee_to_commune["data_coverage_status"] == "fully_complete",
    "insee_commune_id"
].astype(str).str.zfill(5).unique()

complete_communes_set = set(complete_communes)

print("Number of fully complete communes:", len(complete_communes))


Number of fully complete communes: 362


In [ ]:
# F.3 - Create final insee_to_commune dataset

In [267]:
# Reference subset of insee_to_commune
insee_to_commune_complete = insee_to_commune[
    insee_to_commune["insee_commune_id"].astype(str).str.zfill(5).isin(complete_communes_set)
].copy()
insee_to_commune_complete.head()

,insee_commune_id,commune_name_upper,commune_name,has_crime_data,has_income_data_any,has_income_data_all,income_data_coverage,has_air_pollution_data,density_class,density_label,count_agriculture,count_industry,count_construction,count_commercial_services,count_public_services,total_count,economy_data_status,available_source_count,data_coverage_status
2,01004,AMBERIEU EN BUGEY,Ambérieu-en-Bugey,True,True,True,1.0,True,2.0,Urbain intermédiaire,2.0,33.0,57.0,411.0,89.0,592.0,complete,3,fully_complete
29,01033,VALSERHONE,Valserhône,True,True,True,1.0,True,2.0,Urbain intermédiaire,4.0,39.0,95.0,328.0,59.0,525.0,complete,3,fully_complete
30,01034,BELLEY,Belley,True,True,True,1.0,True,2.0,Urbain intermédiaire,4.0,41.0,46.0,258.0,56.0,405.0,complete,3,fully_complete
47,01053,BOURG EN BRESSE,Bourg-en-Bresse,True,True,True,1.0,True,2.0,Urbain intermédiaire,5.0,96.0,104.0,1517.0,304.0,2026.0,complete,3,fully_complete
124,01143,DIVONNE LES BAINS,Divonne-les-Bains,True,True,True,1.0,True,2.0,Urbain intermédiaire,3.0,8.0,9.0,195.0,33.0,248.0,complete,3,fully_complete


In [ ]:
# Keep only needed columns

In [268]:
cols_to_drop = [
    "commune_name_upper",
    "has_crime_data",
    "has_income_data_any",
    "has_income_data_all",
    "has_air_pollution_data", 
    "economy_data_status", 
    "available_source_count", 
    "data_coverage_status"
]

insee_to_commune_complete.drop(
    columns=cols_to_drop,
    inplace=True
)

insee_to_commune_complete.reset_index(drop=True, inplace=True)

insee_to_commune_complete.head()

,insee_commune_id,commune_name,income_data_coverage,density_class,density_label,count_agriculture,count_industry,count_construction,count_commercial_services,count_public_services,total_count
0,01004,Ambérieu-en-Bugey,1.0,2.0,Urbain intermédiaire,2.0,33.0,57.0,411.0,89.0,592.0
1,01033,Valserhône,1.0,2.0,Urbain intermédiaire,4.0,39.0,95.0,328.0,59.0,525.0
2,01034,Belley,1.0,2.0,Urbain intermédiaire,4.0,41.0,46.0,258.0,56.0,405.0
3,01053,Bourg-en-Bresse,1.0,2.0,Urbain intermédiaire,5.0,96.0,104.0,1517.0,304.0,2026.0
4,01143,Divonne-les-Bains,1.0,2.0,Urbain intermédiaire,3.0,8.0,9.0,195.0,33.0,248.0


In [269]:
insee_to_commune_complete.info()

<class 'pandas.DataFrame'>
RangeIndex: 362 entries, 0 to 361
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   insee_commune_id           362 non-null    str    
 1   commune_name               362 non-null    str    
 2   income_data_coverage       362 non-null    float64
 3   density_class              362 non-null    float64
 4   density_label              362 non-null    str    
 5   count_agriculture          362 non-null    float64
 6   count_industry             362 non-null    float64
 7   count_construction         362 non-null    float64
 8   count_commercial_services  362 non-null    float64
 9   count_public_services      362 non-null    float64
 10  total_count                362 non-null    float64
dtypes: float64(8), str(3)
memory usage: 31.2 KB


In [270]:
insee_to_commune_complete.to_csv(
    f"{subset_dir}/insee_to_commune_complete.csv",
    index=False
)

In [ ]:
# F.4 - Create final disposable_income_2021 dataset

In [271]:
disposable_income_2021.head()

,iris_id,commune_name,place_in_commune,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_revenus_patrimoine_pct,part_prestations_sociales_pct,part_prestations_familiales_pct,part_minima_sociaux_pct,part_prestations_logement_pct,part_impots_pct
0,010040101,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,19.0,14990.0,20350.0,26140.0,11620.0,32060.0,70.8,26.9,6.2,8.6,3.3,3.8,1.5,-12.5
1,010040102,Ambérieu-en-Bugey,Longeray-Gare,25.0,13880.0,18570.0,24760.0,10580.0,31130.0,70.6,24.9,5.8,11.1,3.7,5.1,2.3,-12.4
2,010040201,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,19.0,15190.0,20700.0,27180.0,11400.0,34450.0,72.5,27.2,6.4,7.7,2.8,3.3,1.6,-13.8
3,010040202,Ambérieu-en-Bugey,Tiret-Les Allymes,8.0,19600.0,25230.0,33170.0,14810.0,41230.0,73.3,23.8,16.2,4.0,1.8,1.5,0.7,-17.3
4,010330102,Valserhône,Centre Ville,24.0,14050.0,20420.0,29640.0,9410.0,42390.0,78.9,23.7,5.2,5.3,1.5,2.5,1.3,-13.1


In [272]:
# first add a column insee_commune_id = first 5 characters of iris id - add it as first index comumn
disposable_income_2021["insee_commune_id"] = disposable_income_2021["iris_id"].astype(str).str[:5]

# Reorder columns to place insee_commune_id first
cols = ["insee_commune_id"] + [col for col in disposable_income_2021.columns if col != "insee_commune_id"]
disposable_income_2021 = disposable_income_2021[cols]

disposable_income_2021.head()

,insee_commune_id,iris_id,commune_name,place_in_commune,taux_bas_revenus_pct,revenu_q1_eur,revenu_median_eur,revenu_q3_eur,revenu_d1_eur,revenu_d9_eur,part_revenus_activite_pct,part_pensions_pct,part_revenus_patrimoine_pct,part_prestations_sociales_pct,part_prestations_familiales_pct,part_minima_sociaux_pct,part_prestations_logement_pct,part_impots_pct
0,01004,010040101,Ambérieu-en-Bugey,Les Pérouses-Triangle d'Activités,19.0,14990.0,20350.0,26140.0,11620.0,32060.0,70.8,26.9,6.2,8.6,3.3,3.8,1.5,-12.5
1,01004,010040102,Ambérieu-en-Bugey,Longeray-Gare,25.0,13880.0,18570.0,24760.0,10580.0,31130.0,70.6,24.9,5.8,11.1,3.7,5.1,2.3,-12.4
2,01004,010040201,Ambérieu-en-Bugey,Centre-Saint-Germain-Vareilles,19.0,15190.0,20700.0,27180.0,11400.0,34450.0,72.5,27.2,6.4,7.7,2.8,3.3,1.6,-13.8
3,01004,010040202,Ambérieu-en-Bugey,Tiret-Les Allymes,8.0,19600.0,25230.0,33170.0,14810.0,41230.0,73.3,23.8,16.2,4.0,1.8,1.5,0.7,-17.3
4,01033,010330102,Valserhône,Centre Ville,24.0,14050.0,20420.0,29640.0,9410.0,42390.0,78.9,23.7,5.2,5.3,1.5,2.5,1.3,-13.1


In [273]:
disposable_income_2021_complete = disposable_income_2021[
    disposable_income_2021["insee_commune_id"]
    .astype(str)
    .str.zfill(5)
    .isin(complete_communes_set)
].copy()

disposable_income_2021_complete["insee_commune_id"].nunique()

362

In [274]:
disposable_income_2021_complete.to_csv(
    f"{subset_dir}/disposable_income_2021_complete.csv",
    index=False
)

In [ ]:
# F.5 - Create final crime dataset

In [275]:
crime.head()

,insee_commune_id,annee,crime_type,unite_de_compte,nombre,taux_pour_mille,insee_pop_ref,data_available
0,01001,2016,Violences physiques intrafamiliales,Victime,0.0,0.0,767,True
1,01001,2016,Violences physiques hors cadre familial,Victime,NaN,NaN,767,False
2,01001,2016,Violences sexuelles,Victime,0.0,0.0,767,True
3,01001,2016,Vols avec armes,Infraction,0.0,0.0,767,True
4,01001,2016,Vols violents sans arme,Infraction,0.0,0.0,767,True


In [276]:
crime_complete = crime[
    crime["insee_commune_id"]
    .astype(str)
    .str.zfill(5)
    .isin(complete_communes_set)
].copy()

crime_complete["insee_commune_id"].nunique()

362

In [277]:
crime_complete.to_csv(
    f"{subset_dir}/crime_complete.csv",
    index=False
)

In [ ]:
# F.6 - Create final air_pollution dataset

In [278]:
air_pollution_complete = Air_Pollution[
    Air_Pollution["insee_commune_id"].astype(str).str.zfill(5).isin(complete_communes_set)
].copy()

air_pollution_complete["insee_commune_id"].nunique()

362

In [279]:
air_pollution_complete.to_csv(
    f"{subset_dir}/air_pollution_complete.csv",
    index=False
)

In [280]:
# F.7 - Quick control summary
print("insee_to_commune_complete:", insee_to_commune_complete.shape)
print("disposable_income_2021_complete:", disposable_income_2021_complete.shape)
print("crime_complete:", crime_complete.shape)
print("air_pollution_complete:", air_pollution_complete.shape)

insee_to_commune_complete: (362, 11)
disposable_income_2021_complete: (2457, 18)
crime_complete: (48870, 8)
air_pollution_complete: (130463, 17)
